# ADI/OCO — MRC·PSM decorrect → TROCS → WK/RK → CPE38
위에서 아래로 셀을 실행하면서 중간 테이블과 저장 파일을 확인합니다. SQL, 데이터 결합, 보정 함수가 이 노트북 안에 있으므로 직접 수정할 수 있습니다. 별도의 Python 모듈은 필요하지 않습니다.

- **00~15:** 기존 MRC·PSM decorrect. 기존 계산과 변수는 유지합니다.
- **16~19:** 기적용 TROCS RK1~72 조회 → wafer 전체 mark 보정값 계산 → shot/wafer 자료 저장 → 실측 point에 연결.
- **20~21:** ADI/OCO 각각 wafer별 WK20/RK18 동시 회귀 → 제거값과 WK/RK 계수 분리 저장.
- **22~27:** 현재 `013.py` 기준 shot별 CPE38 회귀 → CPE 제거값·shot RK 계수·진단값·최종 residual 저장.
- **28:** 분해식 검증, wafer별/shot별 최종 요약, 저장 파일 목록과 실행 설정 저장.

원본처럼 TROCS는 `trocs_fit_x/y`로 연결하며 `raw_x/y`에서 추가 차감하지 않습니다. `raw → WK/RK 제거 → CPE 제거`로 진행합니다. TROCS의 기적용 `rk1~72_valn`, wafer 회귀 `RK*_X/Y`, shot CPE38 `RK*_X/Y`는 서로 다른 파일에 저장합니다.

모든 행은 Parquet으로 저장합니다. 기본 Excel 미리보기는 최대 10,000행이며, 큰 테이블은 파일명에 `_sample_행수`를 붙입니다. 전체 Excel이 필요하면 00의 `EXCEL_PREVIEW_ROWS`를 늘리세요(Excel 행 제한 적용). 각 셀에서 파일 링크를 표시하며 `saved_files.csv`에 목록을 기록합니다.

live 모드는 사내 `bigdataquery`가 필요합니다. replay는 저장된 조회 스냅샷을 사용합니다. 과거 00~15 스냅샷에는 TROCS 조회가 없으므로 16에서 새로 조회하거나 해당 SQL 결과 파일을 지정해야 합니다. 생산 DB 실행은 사내 환경에서 확인하세요.

**데이터 해석:** ADI의 실제 오차 기준은 `raw_x/raw_y`입니다. `X_reg/Y_reg`는 계측기 출력이며 raw와 구분합니다. 후속 상관/전달 시뮬레이션의 OCO 기준은 사용자 확인에 따라 **OCO Reg**를 사용합니다. 이 노트북의 20~27번 회귀는 이미 raw 기반이므로 이번 설명 변경으로 계산식을 바꾸지 않습니다.


## 00. 라이브러리와 실행 설정
먼저 아래 설정을 수정합니다. `DATA_MODE="replay"`이면 `REPLAY_DIR`에 기존 snapshots 폴더를 지정하세요.


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import hashlib
import json
import logging
import os
import warnings
import numpy as np
import pandas as pd
from IPython.display import display, FileLink

pd.set_option("display.max_columns", 20)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


In [ ]:
@dataclass
class PipelineConfig:
    days: int = 8
    exclude_recent_days: int = 1
    expo_days: int = 10
    target_lotid: list[str] | str | None = None
    target_pstepseq: str | None = "VC075030"
    target_mstepseq: str | None = "VC075040"
    target_mstepseq_oco: str | None = "VC077251"
    db_user: str = "SIMAXP2"
    db_user_apc: str = "EAPCP2"
    slot_min: int = 9
    slot_max: int = 14
    rawdata_gap_seconds: int = 60
    apc_match_seconds: int = 60
    wafer_radius_um: float = 147000.0
    max_query_keys: int = 1000
    rawdata_row_limit: int = 10000000
    reference_date: str | None = None  # Python 측 ADI 날짜 필터용. SQL의 now()는 DB 기준.
    output_dir: str = "output_decorrect_notebook"
    save_parquet: bool = True
    save_excel: bool = False
    save_snapshots: bool = True
    max_excel_rows: int = 1048575

    def __post_init__(self):
        if self.days <= 0 or not 0 <= self.exclude_recent_days < self.days:
            raise ValueError("days > exclude_recent_days >= 0 이어야 합니다.")
        if self.slot_min > self.slot_max or self.max_query_keys <= 0 or self.rawdata_row_limit <= 0:
            raise ValueError("wafer 범위와 조회 제한을 확인하세요.")
        if self.rawdata_gap_seconds < 0 or self.apc_match_seconds < 0:
            raise ValueError("시간 간격은 0 이상이어야 합니다.")

DATA_MODE = "live"  # "live": 사내 DB 조회 / "replay": 저장된 조회 결과 사용
REPLAY_DIR = None  # 예: Path("output_decorrect_notebook/snapshots")
OUTPUT_DIR = Path.cwd() / "output_decorrect_notebook"
SAVE_PARQUET = True
SAVE_EXCEL = True
EXCEL_PREVIEW_ROWS = 10000  # 이보다 큰 테이블은 Excel 샘플 + 전체 Parquet을 저장
PREVIEW_ROWS = 10

cfg = PipelineConfig()
cfg.days = 8
cfg.exclude_recent_days = 1
cfg.expo_days = 10
cfg.target_lotid = None  # 예: ["B0K486.1", "B0K271.1"]
cfg.target_pstepseq = "VC075030"
cfg.target_mstepseq = "VC075040"
cfg.target_mstepseq_oco = "VC077251"
cfg.slot_min = 9
cfg.slot_max = 14
cfg.reference_date = None  # Python ADI 날짜 필터. SQL now()의 날짜를 바꾸지는 않음
cfg.output_dir = str(OUTPUT_DIR)
cfg.save_snapshots = True
cfg.save_parquet = SAVE_PARQUET
cfg.save_excel = SAVE_EXCEL
cfg.__post_init__()
if EXCEL_PREVIEW_ROWS < 1 or PREVIEW_ROWS < 1:
    raise ValueError("미리보기 행 수는 1 이상이어야 합니다.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("중간 결과 저장 폴더:", OUTPUT_DIR.resolve())


## 01. 공통 함수·조회 연결·중간 파일 저장
조회 조건이 같으면 MRC 결과를 재사용합니다. 같은 SQL로 DB를 다시 조회하려면 `source.release_cache()` 후 해당 조회 셀을 재실행하세요. 설정을 바꾸면 이 셀부터 다시 실행합니다.


In [ ]:
LOG = logging.getLogger("decorrect")

def require_columns(df, columns, label):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"{label}: 필수 컬럼 누락 {missing}")

def numeric(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def integer(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64")

def clean_columns(df):
    return df.loc[:, ~df.columns.duplicated()].copy() if df.columns.duplicated().any() else df

def lot_list(value):
    if value is None:
        return []
    if isinstance(value, str):
        value = value.strip().strip("()[]").split(",")
    return [str(v).strip().strip("'\"") for v in value if str(v).strip().strip("'\"")]

def quote(value):
    return "'" + str(value).replace("'", "''") + "'"

def in_filter(column, values, numeric_values=False):
    values = [v for v in values if pd.notna(v)]
    if not values:
        return "AND 1 = 0"  # 빈 IN 조건과 의도하지 않은 전체 조회를 막습니다.
    text = ",".join(str(int(v)) if numeric_values else quote(v) for v in values)
    return f"AND {column} IN ({text})"

def limited_keys(series, limit, label):
    values = series.dropna().unique().tolist()
    if len(values) > limit:
        warnings.warn(f"{label}: {len(values)} keys 중 원본과 동일하게 앞 {limit}개만 조회합니다.", stacklevel=2)
    return values[:limit]

def coord_key(series):
    rounded = pd.to_numeric(series, errors="coerce").round(2)
    return rounded.map(lambda value: f"{value:.2f}" if pd.notna(value) else "<NA>")

def psm_key(lot, chuck, x, y):
    return lot.astype(str).str.strip() + "_" + chuck.astype(str).str.strip() + "_" + coord_key(x) + "_" + coord_key(y)

def chuck_for_slot(series):
    slots = integer(series)
    return pd.Series(np.where(slots.fillna(0) % 2 == 1, "EX1", "EX2"), index=series.index).mask(slots.isna())

LOT_COLUMNS = (
    "l.lot_transn_seq,l.photo_transn_seq,l.impala_insert_time,l.db_user,l.`timestamp`,l.lotid,l.slotid,"
    "l.pstepseq,l.mstepseq,l.ppid,l.peqpid,l.chuckid,l.reticleid,l.steppitch_x,l.steppitch_y,l.mapshift_x,l.mapshift_y,"
    "l.dmargin_x,l.dmargin_y,l.outlr_resdl_spec_ratio,l.chip_x_qty,l.chip_y_qty,l.psm_mmo_ref_eqp_id,l.per_shot_mrc_id"
)

MRC_COLUMNS = [f"wk{i}" for i in range(1, 21)] + [f"rk{i}" for i in range(3, 21)]

PSM_COLUMNS = [f"mrc_k{i}_valn" for i in range(1, 73)]

def equals_filter(column, value):
    return f"AND {column} = {quote(value)}" if value else ""


In [ ]:
class QuerySource:
    def __init__(self, get_data=None, snapshot_dir=None, replay_dir=None):
        self.get_data = get_data
        self.snapshot_dir = Path(snapshot_dir) if snapshot_dir else None
        self.replay_dir = Path(replay_dir) if replay_dir else None
        self.cache = {}
        self.query_count = 0
        self.manifest = {"queries": {}}
        if self.replay_dir:
            self.manifest = json.loads((self.replay_dir / "manifest.json").read_text(encoding="utf-8"))

    def fetch(self, name, sql, reuse=True):
        signature = hashlib.sha256(" ".join(sql.split()).encode()).hexdigest()
        if self.replay_dir:
            saved = self.manifest["queries"].get(name)
            if not saved:
                raise FileNotFoundError(f"조회 스냅샷 없음: {name}. 해당 조건으로 live 실행을 먼저 하세요.")
            if saved["sql_sha256"] != signature:
                raise ValueError(f"{name}: 저장 당시 SQL 조건과 다릅니다. 같은 설정으로 replay하거나 새로 조회하세요.")
            frame = pd.read_parquet(self.replay_dir / f"{name}.parquet")
        elif reuse and signature in self.cache:
            frame = self.cache[signature]
            LOG.info("%s: 동일 SQL 조회 결과 재사용", name)
        else:
            if self.get_data is None:
                try:
                    import bigdataquery as bdq
                except ModuleNotFoundError as exc:
                    raise RuntimeError("live 실행에는 사내 bigdataquery가 필요합니다. 사내 환경에서 실행하거나 --replay로 저장된 스냅샷을 사용하세요.") from exc
                self.get_data = bdq.getData
            frame = self.get_data(sql)
            if not isinstance(frame, pd.DataFrame):
                raise TypeError(f"{name}: getData는 DataFrame을 반환해야 합니다.")
            self.query_count += 1
            if reuse:
                self.cache[signature] = frame
        LOG.info("%s: %s rows", name, len(frame))
        if self.snapshot_dir:
            self.snapshot_dir.mkdir(parents=True, exist_ok=True)
            frame.to_parquet(self.snapshot_dir / f"{name}.parquet", index=False)
            (self.snapshot_dir / f"{name}.sql").write_text(sql.strip() + "\n", encoding="utf-8")
            self.manifest["queries"][name] = {"rows": len(frame), "sql_sha256": signature}
        return frame.copy() if reuse else frame

    def save_manifest(self, cfg):
        if self.snapshot_dir:
            self.manifest["config"] = asdict(cfg)
            self.manifest["query_count"] = self.query_count
            self.manifest["created_at"] = pd.Timestamp.now().isoformat()
            (self.snapshot_dir / "manifest.json").write_text(json.dumps(self.manifest, ensure_ascii=False, indent=2), encoding="utf-8")

    def release_cache(self):
        self.cache.clear()

if DATA_MODE not in ("live", "replay"):
    raise ValueError("DATA_MODE는 live 또는 replay여야 합니다.")
if DATA_MODE == "replay" and REPLAY_DIR is None:
    raise ValueError("replay 모드에는 REPLAY_DIR을 지정하세요.")
snapshot_dir = OUTPUT_DIR / "snapshots" if DATA_MODE == "live" and cfg.save_snapshots else None
source = QuerySource(snapshot_dir=snapshot_dir, replay_dir=REPLAY_DIR if DATA_MODE == "replay" else None)
recorded_date = source.manifest.get("config", {}).get("reference_date")
cfg.reference_date = cfg.reference_date or recorded_date or pd.Timestamp.today().date().isoformat()

def fetch_frame(name, sql, reuse=True):
    frame = source.fetch(name, sql, reuse=reuse)
    source.save_manifest(cfg)  # 끝까지 돌리지 않아도 완료한 조회를 재실행할 수 있게 저장
    return frame

def snapshot_path(name):
    folder = source.replay_dir or source.snapshot_dir
    path = folder / f"{name}.parquet" if folder else None
    return path if path is not None and path.is_file() else None

STAGE_DIR = OUTPUT_DIR / "stages"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
stage_files = {}

def save_stage(name, frame, columns=None, existing_parquet=None):
    print(f"[{name}] {len(frame):,} rows × {len(frame.columns)} columns")
    selected = [col for col in columns if col in frame] if columns else list(frame.columns)
    display(frame[selected].head(PREVIEW_ROWS))
    paths = []
    if SAVE_PARQUET:
        path = Path(existing_parquet) if existing_parquet is not None else STAGE_DIR / f"{name}.parquet"
        if existing_parquet is None:
            frame.to_parquet(path, index=False)
        paths.append(("parquet_full", path, len(frame)))
    if SAVE_EXCEL:
        limit = min(EXCEL_PREVIEW_ROWS, cfg.max_excel_rows)
        exported = frame.head(limit)
        suffix = f"_sample_{len(exported)}" if len(frame) > limit else ""
        path = STAGE_DIR / f"{name}{suffix}.xlsx"
        exported.to_excel(path, index=False, engine="openpyxl")
        paths.append(("excel_sample" if suffix else "excel_full", path, len(exported)))
    for kind, path, exported_rows in paths:
        key = f"{name}:{kind}"
        stage_files[key] = {"stage": name, "kind": kind, "total_rows": len(frame), "exported_rows": exported_rows, "path": str(path.resolve())}
        display(FileLink(os.path.relpath(path, Path.cwd()), result_html_prefix=f"{kind} ({exported_rows:,}행): "))
    pd.DataFrame(stage_files.values()).to_csv(OUTPUT_DIR / "saved_files.csv", index=False, encoding="utf-8-sig")

print("데이터 모드:", DATA_MODE, "| ADI 기준일:", cfg.reference_date)


## 02. OCO LOTINFO 조회
`sql_lotinfo_oco`와 `df_lotinfo_oco_query`를 확인합니다. 이 셀의 SQL 함수에서 OCO 조회 조건을 수정할 수 있습니다.


In [ ]:
def lotinfo_sql(cfg, mode, photo_ids=None):
    extra_days = 5 if mode == "OCO" else 10
    recent_days = max(0, cfg.exclude_recent_days - 3) if mode == "OCO" else cfg.exclude_recent_days
    metro = cfg.target_mstepseq_oco if mode == "OCO" else cfg.target_mstepseq
    selection = in_filter("l.lotid", lot_list(cfg.target_lotid)) if mode == "OCO" and cfg.target_lotid else ""
    if mode == "ADI":
        selection = in_filter("l.photo_transn_seq", photo_ids or [], True)
    row_limit = 1000 if mode == "OCO" else 100000
    return f"""
WITH lotinfo_raw AS (
  SELECT * FROM (
    SELECT {LOT_COLUMNS}, row_number() OVER (PARTITION BY l.lotid, l.slotid ORDER BY l.impala_insert_time DESC) AS rn
    FROM ees_ds_eai.apc_nautil_dat_lotinfo l
    WHERE l.impala_insert_time >= now() - interval {cfg.days + extra_days} days
      AND l.`timestamp` >= now() - interval {cfg.days + extra_days} days
      AND l.`timestamp` <= now() - interval {recent_days} days
      AND l.db_user = {quote(cfg.db_user)}
      {selection}
      {equals_filter('l.pstepseq', cfg.target_pstepseq)}
      {equals_filter('l.mstepseq', metro)}
  ) t WHERE t.rn = 1
)
SELECT * FROM lotinfo_raw LIMIT {row_limit}
"""

sql_lotinfo_oco = lotinfo_sql(cfg, "OCO")
df_lotinfo_oco_query = fetch_frame("lotinfo_oco", sql_lotinfo_oco)
save_stage("02_LOTINFO_OCO_QUERY", df_lotinfo_oco_query, existing_parquet=snapshot_path("lotinfo_oco"))


## 03. 동일 노광 이력의 ADI LOTINFO 조회
OCO의 photo_transn_seq를 이용합니다. 원본처럼 최대 조회 key 수를 유지하며 초과 시 알려줍니다.


In [ ]:
photo_ids = limited_keys(df_lotinfo_oco_query["photo_transn_seq"], cfg.max_query_keys, "ADI LOTINFO") if not df_lotinfo_oco_query.empty else []
sql_lotinfo_adi = lotinfo_sql(cfg, "ADI", photo_ids)
df_lotinfo_adi_query = fetch_frame("lotinfo_adi", sql_lotinfo_adi) if photo_ids else pd.DataFrame()
save_stage("03_LOTINFO_ADI_QUERY", df_lotinfo_adi_query, existing_parquet=snapshot_path("lotinfo_adi"))


## 04. 날짜·동일 wafer·slot 범위 적용
`df_lotinfo_adi`, `df_lotinfo_oco`가 이후의 분석 대상입니다. 대상 선정 로직은 이 셀에서 수정합니다.


In [ ]:
def select_lot_pair(adi, oco, cfg):
    if adi.empty or oco.empty:
        return adi.iloc[:0].copy(), oco.iloc[:0].copy(), pd.DataFrame()
    adi, oco = adi.copy(), oco.copy()
    for frame in (adi, oco):
        require_columns(frame, ["photo_transn_seq", "slotid", "lot_transn_seq", "lotid", "timestamp"], "LOTINFO")
        frame["photo_transn_seq"], frame["slotid"] = integer(frame["photo_transn_seq"]), integer(frame["slotid"])
    adi["timestamp"] = pd.to_datetime(adi["timestamp"], errors="coerce")
    today = pd.Timestamp(cfg.reference_date).normalize() if cfg.reference_date else pd.Timestamp.today().normalize()
    start, end = today - pd.Timedelta(days=cfg.days), today - pd.Timedelta(days=cfg.exclude_recent_days)
    adi = adi[adi["timestamp"].ge(start) & adi["timestamp"].lt(end)].copy()
    keys = ["photo_transn_seq", "slotid"]
    common = oco[keys].dropna().drop_duplicates().merge(adi[keys].dropna().drop_duplicates(), on=keys, how="inner")
    adi, oco = [frame.merge(common, on=keys, how="inner").sort_values(keys).reset_index(drop=True) for frame in (adi, oco)]
    adi, oco = [frame[frame["slotid"].between(cfg.slot_min, cfg.slot_max)].copy() for frame in (adi, oco)]
    for frame in (adi, oco):
        frame["slotid"] = frame["slotid"].astype(int)
    oco_extra = oco[keys + ["lot_transn_seq", "timestamp", "mstepseq"]].drop_duplicates(keys)
    oco_extra = oco_extra.rename(columns={"lot_transn_seq": "lot_transn_seq_oco", "timestamp": "timestamp_oco", "mstepseq": "mstepseq_oco"})
    saved = adi.merge(oco_extra, on=keys, how="left")
    if "per_shot_mrc_id" in saved:
        parts = saved["per_shot_mrc_id"].astype(str).str.split("/").str[-1].str.split("_")
        saved["base_eqpid"] = parts.str[0]
        saved["psm_update_time"] = parts.str[1].astype("string").str.replace(r"^20", "", regex=True).str[:10]
    return adi, oco, saved

df_lotinfo_adi, df_lotinfo_oco, df_lotinfo_save = select_lot_pair(df_lotinfo_adi_query, df_lotinfo_oco_query, cfg)
save_stage("04_LOTINFO_ADI_SELECTED", df_lotinfo_adi)
save_stage("04_LOTINFO_OCO_SELECTED", df_lotinfo_oco)
save_stage("04_LOTINFO_PAIRED", df_lotinfo_save)


## 05. RAWDATA / TESTDATA / PERSHOT 조회
가공 전 조회 결과를 확인합니다. 큰 조회 결과는 snapshots의 전체 Parquet을 연결해서 중복 저장을 줄입니다.


In [ ]:
def rawdata_sql(cfg, lots):
    ids = limited_keys(lots["lot_transn_seq"], cfg.max_query_keys, "RAWDATA")
    return f"""
SELECT r.lot_transn_seq,r.slot_id,r.transn_occur_date,r.data_kind,r.test_point_no,r.coordinate_x,r.coordinate_y,r.value_x,r.value_y,r.mrc_x_valn,r.mrc_y_valn
FROM ees_ds_eai.apc_nautil_dat_rawdata r
WHERE r.impala_insert_time >= now() - interval {cfg.days + 10} days AND r.db_user = {quote(cfg.db_user)}
  {in_filter('r.lot_transn_seq', ids, True)} AND r.data_kind IN ('RAWDATA','TESTDATA','PERSHOT')
LIMIT {cfg.rawdata_row_limit}
"""

sql_rawdata_oco = rawdata_sql(cfg, df_lotinfo_oco) if not df_lotinfo_oco.empty else None
sql_rawdata_adi = rawdata_sql(cfg, df_lotinfo_adi) if not df_lotinfo_adi.empty else None
df_rawdata_oco_query = fetch_frame("rawdata_oco", sql_rawdata_oco, reuse=False) if sql_rawdata_oco else pd.DataFrame()
df_rawdata_adi_query = fetch_frame("rawdata_adi", sql_rawdata_adi, reuse=False) if sql_rawdata_adi else pd.DataFrame()
save_stage("05_RAWDATA_OCO_QUERY", df_rawdata_oco_query, existing_parquet=snapshot_path("rawdata_oco"))
save_stage("05_RAWDATA_ADI_QUERY", df_rawdata_adi_query, existing_parquet=snapshot_path("rawdata_adi"))


## 06. 종류별 최신 batch 선택
RAWDATA / TESTDATA / PERSHOT 각각 마지막 batch를 남깁니다. 원본 조회와 선택 후의 행 수를 비교하세요.


In [ ]:
def latest_raw_batch(raw, lots, gap_seconds=60):
    if raw.empty or lots.empty:
        return raw.iloc[:0].copy()
    require_columns(raw, ["lot_transn_seq", "slot_id", "data_kind", "transn_occur_date"], "RAWDATA")
    df = raw.copy()
    df["transn_occur_date"] = pd.to_datetime(df["transn_occur_date"].astype(str), errors="coerce")
    df["slot_id"] = integer(df["slot_id"])
    df = df.dropna(subset=["slot_id", "transn_occur_date"]).copy()
    df["slot_id"] = df["slot_id"].astype(int)
    keys = lots[["lot_transn_seq", "slotid"]].drop_duplicates().copy()
    keys["lot_transn_seq"], keys["slotid"] = integer(keys["lot_transn_seq"]), integer(keys["slotid"])
    df["lot_transn_seq"] = integer(df["lot_transn_seq"])
    df = df.merge(keys.dropna(), left_on=["lot_transn_seq", "slot_id"], right_on=["lot_transn_seq", "slotid"], how="inner")
    groups = ["lot_transn_seq", "slot_id", "data_kind"]
    df = df.sort_values(groups + ["transn_occur_date"]).reset_index(drop=True)
    previous = df.groupby(groups)["transn_occur_date"].shift()
    starts = previous.isna() | (df["transn_occur_date"] - previous).dt.total_seconds().gt(gap_seconds)
    df["_batch"] = starts.astype(int).groupby([df[col] for col in groups]).cumsum()
    # transform으로 최대 batch를 붙이면 원본 merge와 같은 행 선택을 중간 테이블 없이 수행합니다.
    last = df.groupby(groups)["_batch"].transform("max")
    return df[df["_batch"].eq(last)].drop(columns="_batch").reset_index(drop=True)

df_rawdata_oco = latest_raw_batch(df_rawdata_oco_query, df_lotinfo_oco, cfg.rawdata_gap_seconds)
df_rawdata_adi = latest_raw_batch(df_rawdata_adi_query, df_lotinfo_adi, cfg.rawdata_gap_seconds)
save_stage("06_RAWDATA_OCO_LATEST", df_rawdata_oco)
save_stage("06_RAWDATA_ADI_LATEST", df_rawdata_adi)
source.release_cache()


## 07. EXPO·F1 공정 정보 조회
노광 이력의 보정 index와 장비 정보를 가져옵니다. `df_paramdata`는 최신 F1 선택 전의 조회 결과입니다.


In [ ]:
def expo_sql(cfg, lots):
    ids = limited_keys(lots["lotid"], cfg.max_query_keys, "EXPO")
    return f"""
SELECT CAST(e.apc_hist_index_no AS BIGINT) AS apc_hist_index_no,e.apc_trocs_hist_index_no,e.impala_insert_time,e.seq_no,e.lot_id,e.slot_id,e.photo_step_seq,e.metro_step_seq,e.photo_date,e.photo_transn_seq,e.mmo_mrc_ref_eqp_id
FROM ees_ds_eai.apc_expo_overlay_lot e
WHERE e.impala_insert_time >= now() - interval {cfg.expo_days + 5} days AND e.db_user = {quote(cfg.db_user)}
  {in_filter('e.lot_id', ids)} {equals_filter('e.photo_step_seq', cfg.target_pstepseq)}
LIMIT 100000
"""

def f1_sql(cfg, lots):
    ids = limited_keys(lots["photo_transn_seq"], cfg.max_query_keys, "F1")
    return f"""
SELECT p.photo_transn_seq,p.lotid,p.slotid,p.base_eqp_id1,p.impala_insert_time
FROM ees_ds_eai.apc_nautil_paramdata p
WHERE p.impala_insert_time >= now() - interval {cfg.days + 5} days AND p.db_user = {quote(cfg.db_user)}
  {in_filter('p.photo_transn_seq', ids, True)} AND p.subname = 'F1' {equals_filter('p.mstepseq', cfg.target_mstepseq)}
LIMIT 1000000
"""

def prepare_expo(expo, lots):
    if expo.empty or lots.empty:
        return expo.iloc[:0].copy()
    expo = expo.sort_values("photo_date").drop_duplicates(["lot_id", "slot_id"], keep="last").copy()
    expo["slot_id"] = integer(expo["slot_id"])
    return expo.merge(lots[["lotid", "slotid"]].drop_duplicates(), left_on=["lot_id", "slot_id"], right_on=["lotid", "slotid"], how="inner").sort_values(["photo_transn_seq", "slotid"]).reset_index(drop=True)

sql_expo = expo_sql(cfg, df_lotinfo_adi) if not df_lotinfo_adi.empty else None
sql_f1 = f1_sql(cfg, df_lotinfo_adi) if not df_lotinfo_adi.empty else None
df_expo_overlay_query = fetch_frame("expo", sql_expo) if sql_expo else pd.DataFrame()
df_paramdata = fetch_frame("f1", sql_f1) if sql_f1 else pd.DataFrame()
df_expo_overlay = prepare_expo(df_expo_overlay_query, df_lotinfo_adi)
save_stage("07_EXPO_SELECTED", df_expo_overlay)
save_stage("07_F1_QUERY", df_paramdata, existing_parquet=snapshot_path("f1"))


## 08. 측정값·좌표·메타데이터 결합
그룹, 좌표, ID, edge 여부를 계산합니다. `df_adi_measured`, `df_oco_measured`는 MRC decorrect 전 단계이며 원래 X_reg/Y_reg가 남아 있습니다.


In [ ]:
RENAME = {
    "pstepseq": "STEPSEQ", "lotid": "LOT_ID", "slotid": "Wafer", "peqpid": "P_EQPID", "ppid": "Photo_PPID",
    "chuckid": "ChuckID", "reticleid": "ReticleID", "steppitch_x": "STEP_PITCH_X", "steppitch_y": "STEP_PITCH_Y",
    "mapshift_x": "MAP_SHIFT_X", "mapshift_y": "MAP_SHIFT_Y", "dmargin_x": "Dmargin_X", "dmargin_y": "Dmargin_Y",
    "outlr_resdl_spec_ratio": "Outlier_Spec_Ratio", "chip_x_qty": "CHIP_X_NUM", "chip_y_qty": "CHIP_Y_NUM",
    "mmo_mrc_ref_eqp_id": "MMO_MRC_EQP", "mstepseq": "M_STEPSEQ", "timestamp": "M_TIME",
}

FINAL_COLUMNS = [
    "UNIQUE_ID", "UNIQUE_ID2", "UNIQUE_ID3", "UNIQUE_ID4", "STEPSEQ", "LOT_ID", "Wafer",
    "P_EQPID", "Photo_PPID", "P_TIME", "M_TIME", "ChuckID", "ReticleID", "Base_EQP1", "MMO_MRC_EQP", "GROUP", "TEST", "DieX", "DieY",
    "STEP_PITCH_X", "STEP_PITCH_Y", "MAP_SHIFT_X", "MAP_SHIFT_Y", "coordinate_X", "coordinate_Y",
    "fcp_x", "fcp_y", "wf_x", "wf_y", "radius", "CHIP_X_NUM", "CHIP_Y_NUM", "Outlier_Spec_Ratio", "Dmargin_X", "Dmargin_Y",
    "X_reg", "Y_reg", "MRC_RX", "MRC_RY", "PSM_X", "PSM_Y", "Is_Edge_Shot", "Max_Corner_Distance",
    "M_STEPSEQ", "photo_transn_seq", "apc_hist_index_no", "apc_trocs_hist_index_no",
]

def assemble_measurements(raw, lots, expo, f1, mode, cfg):
    if raw.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)
    df = raw[raw["data_kind"].eq("RAWDATA")].rename(columns={"test_point_no": "TEST", "coordinate_x": "DieX", "coordinate_y": "DieY", "value_x": "X_reg", "value_y": "Y_reg"}).copy()
    if df.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)
    df = clean_columns(df).merge(lots, left_on=["lot_transn_seq", "slot_id"], right_on=["lot_transn_seq", "slotid"], how="left", validate="m:1", suffixes=("", "_lotinfo"))
    test = raw[raw["data_kind"].eq("TESTDATA")].rename(columns={"test_point_no": "TEST", "value_x": "coordinate_X", "value_y": "coordinate_Y", "mrc_x_valn": "MRC_RX", "mrc_y_valn": "MRC_RY"})
    if not test.empty:
        columns = ["lot_transn_seq", "slot_id", "TEST", "coordinate_X", "coordinate_Y", "MRC_RX", "MRC_RY"]
        df = df.merge(test[columns], on=columns[:3], how="left", validate="m:1", suffixes=("", "_test"))
    for col in ["coordinate_X", "coordinate_Y", "MRC_RX", "MRC_RY"]:
        if col not in df:
            df[col] = np.nan
    if mode == "ADI":
        pershot = raw[raw["data_kind"].eq("PERSHOT")].rename(columns={"test_point_no": "TEST", "coordinate_x": "DieX", "coordinate_y": "DieY", "value_x": "PSM_X", "value_y": "PSM_Y"})
        columns = ["lot_transn_seq", "slot_id", "TEST", "DieX", "DieY", "PSM_X", "PSM_Y"]
        if not pershot.empty:
            df = df.merge(pershot[columns], on=columns[:5], how="left", validate="m:1", suffixes=("", "_psm"))
        for col in ["PSM_X", "PSM_Y"]:
            if col not in df:
                df[col] = np.nan
    if not expo.empty:
        columns = ["lotid", "slot_id", "photo_transn_seq", "photo_date", "apc_hist_index_no", "apc_trocs_hist_index_no", "mmo_mrc_ref_eqp_id"]
        df = df.merge(expo[columns].rename(columns={"photo_date": "P_TIME", "photo_transn_seq": "photo_transn_seq_expo"}), on=["lotid", "slot_id"], how="left", validate="m:1")
    else:
        for col in ["P_TIME", "apc_hist_index_no", "apc_trocs_hist_index_no", "mmo_mrc_ref_eqp_id"]:
            df[col] = pd.NA
    # 원본의 photo_transn_seq_x/y 충돌 때문에 빠지던 F1 메타데이터만 명시적으로 연결합니다.
    if not f1.empty:
        f1 = f1.copy()
        f1["photo_transn_seq"], f1["slotid"] = integer(f1["photo_transn_seq"]), integer(f1["slotid"])
        f1["_f1_lot_key"] = f1["lotid"].astype(str).str.strip()
        df["_f1_lot_key"] = df["lotid"].astype(str).str.strip()
        keys = ["photo_transn_seq", "_f1_lot_key", "slotid"]
        f1 = f1.sort_values("impala_insert_time").drop_duplicates(keys, keep="last") if "impala_insert_time" in f1 else f1.drop_duplicates(keys, keep="last")
        df = df.merge(f1[keys + ["base_eqp_id1"]], on=keys, how="left", validate="m:1").rename(columns={"base_eqp_id1": "Base_EQP1"})
        df = df.drop(columns="_f1_lot_key")
    df = clean_columns(df.rename(columns=RENAME)).reset_index(drop=True)
    numeric(df, ["TEST", "X_reg", "Y_reg", "PSM_X", "PSM_Y", "MRC_RX", "MRC_RY"])
    df["GROUP"] = pd.cut(df["TEST"], bins=[0, 80, 160, 240], labels=["E1", "E2", "E3"])
    if mode == "OCO":
        group = "E2" if (cfg.target_mstepseq_oco or "")[2:5] == "077" else "E1"
        df = df[df["GROUP"].eq(group)].copy()
    return add_geometry_and_ids(df, cfg)

def add_geometry_and_ids(df, cfg):
    if df.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)
    df = df.reset_index(drop=True)
    columns = ["DieX", "DieY", "STEP_PITCH_X", "STEP_PITCH_Y", "MAP_SHIFT_X", "MAP_SHIFT_Y", "coordinate_X", "coordinate_Y"]
    require_columns(df, columns, "좌표 계산")
    for col in columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors="coerce")
    df["fcp_x"] = df["DieX"] * df["STEP_PITCH_X"] + df["MAP_SHIFT_X"]
    df["fcp_y"] = df["DieY"] * df["STEP_PITCH_Y"] + df["MAP_SHIFT_Y"]
    df["wf_x"], df["wf_y"] = df["fcp_x"] + df["coordinate_X"], df["fcp_y"] + df["coordinate_Y"]
    df["radius"] = np.sqrt(df["wf_x"] ** 2 + df["wf_y"] ** 2)
    fields = ["STEPSEQ", "P_EQPID", "Photo_PPID", "MMO_MRC_EQP", "P_TIME", "M_TIME", "LOT_ID", "Wafer"]
    text = {col: df[col].astype("string").fillna("") for col in fields + ["GROUP", "TEST", "DieX", "DieY"]}
    base = text[fields[0]]
    for col in fields[1:]:
        base = base + "_" + text[col]
    df["UNIQUE_ID3"], df["UNIQUE_ID"] = base, base + "_" + text["GROUP"]
    df["UNIQUE_ID2"] = base + "_" + text["TEST"] + "_" + text["DieX"] + "_" + text["DieY"] + "_" + text["GROUP"]
    df["UNIQUE_ID4"] = base + "_" + text["DieX"] + "_" + text["DieY"] + "_" + text["GROUP"]
    x, y = df["fcp_x"].to_numpy(float), df["fcp_y"].to_numpy(float)
    sx, sy = df["STEP_PITCH_X"].to_numpy(float) / 2, df["STEP_PITCH_Y"].to_numpy(float) / 2
    distances = [(x + dx * sx) ** 2 + (y + dy * sy) ** 2 for dx, dy in [(-1, -1), (-1, 1), (1, -1), (1, 1)]]
    df["Max_Corner_Distance"] = np.sqrt(np.maximum.reduce(distances))
    df["Is_Edge_Shot"] = df["Max_Corner_Distance"].gt(cfg.wafer_radius_um)
    return df[[col for col in FINAL_COLUMNS if col in df]].copy()

df_adi_measured = assemble_measurements(df_rawdata_adi, df_lotinfo_adi, df_expo_overlay, df_paramdata, "ADI", cfg)
df_oco_measured = assemble_measurements(df_rawdata_oco, df_lotinfo_oco, df_expo_overlay, df_paramdata, "OCO", cfg)
save_stage("08_ADI_MEASURED", df_adi_measured)
save_stage("08_OCO_MEASURED", df_oco_measured)


## 09. 기적용 MRC(CMP) 계수 조회·최신 이력 선택
원본처럼 ADI와 OCO 모두 ADI mstepseq의 CMP 계수를 사용합니다. 현재 mrc_sql의 조건은 여기서 수정합니다.


In [ ]:
def mrc_sql(cfg, frame):
    # 원본 007.py와 같이 ADI/OCO 모두 ADI mstepseq의 CMP 계수를 사용합니다.
    ids = limited_keys(frame["photo_transn_seq"], cfg.max_query_keys, "CMP")
    columns = ",".join(f"p.{col}" for col in MRC_COLUMNS)
    return f"""
SELECT p.photo_transn_seq,p.lotid,p.slotid,p.subname,{columns},p.impala_insert_time
FROM ees_ds_eai.apc_nautil_paramdata p
WHERE p.impala_insert_time >= now() - interval {cfg.days + 10} days AND p.db_user = {quote(cfg.db_user)}
  {in_filter('p.photo_transn_seq', ids, True)} AND p.subname = 'CMP' {equals_filter('p.mstepseq', cfg.target_mstepseq)}
LIMIT 100000
"""

def prepare_mrc_input(mrc):
    if mrc.empty:
        return pd.DataFrame(columns=["photo_transn_seq", "lotid", "slotid"] + MRC_COLUMNS)
    require_columns(mrc, ["photo_transn_seq", "lotid", "slotid", "impala_insert_time"] + MRC_COLUMNS, "CMP")
    df = mrc.copy()
    df = df.sort_values("impala_insert_time").drop_duplicates(["photo_transn_seq", "lotid", "slotid"], keep="last")
    df["photo_transn_seq"], df["slotid"] = integer(df["photo_transn_seq"]), integer(df["slotid"])
    return numeric(df, MRC_COLUMNS)

sql_kmrc_adi = mrc_sql(cfg, df_adi_measured) if not df_adi_measured.empty else None
sql_kmrc_oco = mrc_sql(cfg, df_oco_measured) if not df_oco_measured.empty else None
df_kmrc_adi_query = fetch_frame("mrc_adi", sql_kmrc_adi) if sql_kmrc_adi else pd.DataFrame()
df_kmrc_oco_query = fetch_frame("mrc_oco", sql_kmrc_oco) if sql_kmrc_oco else pd.DataFrame()
df_kmrc_adi = prepare_mrc_input(df_kmrc_adi_query)
df_kmrc_oco = prepare_mrc_input(df_kmrc_oco_query)
save_stage("09_MRC_ADI_INPUT", df_kmrc_adi)
save_stage("09_MRC_OCO_INPUT", df_kmrc_oco)


## 10. MRC decorrect
기적용 MRC를 원본의 항 순서·스케일·부호로 계산합니다. X_reg_demrc/Y_reg_demrc와 mrc_status를 확인하세요. 새로운 회귀를 하는 단계는 아닙니다.
기존 부호를 유지합니다: `mrc_fit = -B @ K_applied`, `Reg_demrc = Reg - mrc_fit`. 계측 출력 Reg를 실제 ADI 오차로 바로 해석하지 않습니다.


In [ ]:
def osr_wk20p_rk20p(x, y, rx, ry):
    X_dx = np.vstack([
        np.ones(len(x)),
        x / 1e6, -y / 1e6,
        (x ** 2) / 1e12, (x * y) / 1e12, (y ** 2) / 1e12,
        (x ** 3) / 1e15, (x ** 2 * y) / 1e15, (x * y ** 2) / 1e15, (y ** 3) / 1e15,
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12
    ]).T

    X_dy = np.vstack([
        np.ones(len(y)),
        y / 1e6, x / 1e6,
        (y ** 2) / 1e12, (y * x) / 1e12, (x ** 2) / 1e12,
        (y ** 3) / 1e15, (y ** 2 * x) / 1e15, (y * x ** 2) / 1e15, (x ** 3) / 1e15,
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12, (rx ** 3) / 1e12
    ]).T

    return X_dx, X_dy

MRC_X_COLUMNS = [f"wk{i}" for i in range(1, 21, 2)] + [f"rk{i}" for i in range(3, 21, 2)]

MRC_Y_COLUMNS = [f"wk{i}" for i in range(2, 21, 2)] + [f"rk{i}" for i in range(4, 21, 2)]

def apply_mrc(frame, mrc):
    df = frame.copy()
    for col in ["mrc_fit_x", "mrc_fit_y", "X_reg_demrc", "Y_reg_demrc"]:
        df[col] = np.nan
    df["mrc_status"] = "missing_mrc"
    if df.empty:
        return df
    require_columns(df, ["photo_transn_seq", "LOT_ID", "Wafer", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y", "X_reg", "Y_reg"], "MRC 적용")
    numeric(df, ["X_reg", "Y_reg", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"])
    if not mrc.empty:
        require_columns(mrc, ["photo_transn_seq", "lotid", "slotid"] + MRC_COLUMNS, "MRC 계수")
        # 계수를 모든 측정행에 merge하면 큰 복제 테이블이 생기므로 wafer별 lookup만 만듭니다.
        keys = list(zip(integer(mrc["photo_transn_seq"]), mrc["lotid"], integer(mrc["slotid"])))
        bx = np.ascontiguousarray(mrc[MRC_X_COLUMNS].to_numpy(float))
        by = np.ascontiguousarray(mrc[MRC_Y_COLUMNS].to_numpy(float))
        lookup = {key: i for i, key in reversed(list(enumerate(keys))) if all(pd.notna(v) for v in key)}
        group_keys = pd.DataFrame({"photo": integer(df["photo_transn_seq"]), "lot": df["LOT_ID"], "slot": integer(df["Wafer"])}, index=df.index)
        coordinates = [df[col].to_numpy(float) for col in ["fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"]]
        fit_x, fit_y, status = np.full(len(df), np.nan), np.full(len(df), np.nan), df["mrc_status"].to_numpy(object)
        for key, positions in group_keys.groupby(["photo", "lot", "slot"], sort=False).indices.items():
            idx = lookup.get(key)
            if idx is None:
                continue
            if np.isnan(bx[idx]).any() or np.isnan(by[idx]).any():
                status[positions] = "invalid_mrc_k"
                continue
            mx, my = osr_wk20p_rk20p(*(values[positions] for values in coordinates))
            fit_x[positions], fit_y[positions] = mx.dot(bx[idx]) * -1, my.dot(by[idx]) * -1
            status[positions] = "matched"
        df["mrc_fit_x"], df["mrc_fit_y"], df["mrc_status"] = fit_x, fit_y, status
    df["X_reg_demrc"], df["Y_reg_demrc"] = df["X_reg"] - df["mrc_fit_x"], df["Y_reg"] - df["mrc_fit_y"]
    return df

df_adi_demrc = apply_mrc(df_adi_measured, df_kmrc_adi)
df_oco_demrc = apply_mrc(df_oco_measured, df_kmrc_oco)
mrc_preview = ["LOT_ID", "Wafer", "TEST", "X_reg", "Y_reg", "mrc_fit_x", "mrc_fit_y", "X_reg_demrc", "Y_reg_demrc", "mrc_status"]
save_stage("10_ADI_DEMRC", df_adi_demrc, columns=mrc_preview)
save_stage("10_OCO_DEMRC", df_oco_demrc, columns=mrc_preview)


## 11. APC 일반 이력·EEXY 연결
EEXY와 apc_trocs_hist_index_no는 원본의 PSM 계수 조회에 필요한 연결 정보입니다. TROCS 계수 평가를 수행하지는 않습니다.


In [ ]:
def history_sql(cfg, lots, eexy=False):
    ids = limited_keys(lots["lotid"], cfg.max_query_keys, "APC LOT HIST")
    columns = "CAST(apc_hist_index_no AS BIGINT) AS apc_trocs_hist_index_no,lot_id,step_seq,eqp_id,sub_eqp_id,tkin_tsdt,event_tsdt AS event_tsdt_eexy"
    if not eexy:
        columns = "CAST(apc_hist_index_no AS BIGINT) AS apc_hist_index_no,apc_group_name,lot_id,step_seq,eqp_id,sub_eqp_id,ppid,reticle_id,run_type,event_tsdt,tkin_tsdt"
    condition = "=" if eexy else "!="
    return f"""
SELECT {columns} FROM ees_ds_eai.apc_apc_lot_hist
WHERE db_user = {quote(cfg.db_user_apc)} AND event_tsdt >= now() - interval {cfg.days + 5} days
  AND event_tsdt <= now() - interval {cfg.exclude_recent_days} days AND apc_group_name {condition} 'PHOTO_EEXY_GRP' AND save_type != 'A'
  {in_filter('lot_id', ids)} {equals_filter('step_seq', cfg.target_pstepseq)}
"""

HISTORY_KEYS = ["lot_id", "step_seq", "eqp_id", "sub_eqp_id"]

HISTORY_COLUMNS = ["apc_hist_index_no", "apc_trocs_hist_index_no", "lot_id", "step_seq", "eqp_id", "sub_eqp_id", "ppid", "reticle_id", "run_type", "event_tsdt", "tkin_tsdt"]

PSM_META = ["event_tsdt", "tkin_tsdt", "apc_hist_index_no", "lot_id", "step_seq", "eqp_id", "sub_eqp_id", "ppid", "reticle_id", "run_type", "pos_x_valn", "pos_y_valn"]

def match_apc_history(main, eexy, tolerance_seconds=60):
    if main.empty:
        return pd.DataFrame(columns=HISTORY_COLUMNS)
    main, eexy = main.copy(), eexy.copy()
    if eexy.empty:
        out = main
        out["apc_trocs_hist_index_no"] = pd.NA
    else:
        for frame in (main, eexy):
            for col in HISTORY_KEYS:
                frame[col] = frame[col].astype(str).str.strip()
            frame["tkin_tsdt"] = pd.to_datetime(frame["tkin_tsdt"].astype(str), errors="coerce")
        main["event_tsdt"] = pd.to_datetime(main["event_tsdt"].astype(str), errors="coerce")
        eexy["event_tsdt_eexy"] = pd.to_datetime(eexy["event_tsdt_eexy"].astype(str), errors="coerce")
        main["apc_hist_index_no"], eexy["apc_trocs_hist_index_no"] = integer(main["apc_hist_index_no"]), integer(eexy["apc_trocs_hist_index_no"])
        left = main.dropna(subset=["tkin_tsdt"]).sort_values("tkin_tsdt")
        right = eexy.dropna(subset=["tkin_tsdt"]).sort_values("event_tsdt_eexy").drop_duplicates(HISTORY_KEYS + ["tkin_tsdt"], keep="last").sort_values("tkin_tsdt")
        out = pd.merge_asof(left, right[HISTORY_KEYS + ["tkin_tsdt", "apc_trocs_hist_index_no"]], on="tkin_tsdt", by=HISTORY_KEYS, tolerance=pd.Timedelta(seconds=tolerance_seconds), direction="nearest")
        invalid = main[main["tkin_tsdt"].isna()].copy()
        if not invalid.empty:
            invalid["apc_trocs_hist_index_no"] = pd.NA
            out = pd.concat([out, invalid], ignore_index=True)
    out["event_tsdt"] = pd.to_datetime(out["event_tsdt"], errors="coerce")
    out["apc_hist_index_no"], out["apc_trocs_hist_index_no"] = integer(out["apc_hist_index_no"]), integer(out["apc_trocs_hist_index_no"])
    # rich process key로 바꾸지 않습니다. 원본은 LOT/chuck의 마지막 이력을 선택합니다.
    return out.sort_values("event_tsdt").drop_duplicates(["lot_id", "sub_eqp_id"], keep="last").reset_index(drop=True)

sql_lot_hist_main = history_sql(cfg, df_lotinfo_adi) if not df_lotinfo_adi.empty else None
sql_lot_hist_eexy = history_sql(cfg, df_lotinfo_adi, eexy=True) if not df_lotinfo_adi.empty else None
df_lot_hist_main = fetch_frame("apc_main", sql_lot_hist_main) if sql_lot_hist_main else pd.DataFrame()
df_lot_hist_eexy = fetch_frame("apc_eexy", sql_lot_hist_eexy) if sql_lot_hist_eexy else pd.DataFrame()
df_lot_hist_apc = match_apc_history(df_lot_hist_main, df_lot_hist_eexy, cfg.apc_match_seconds)
save_stage("11_APC_MAIN_QUERY", df_lot_hist_main, existing_parquet=snapshot_path("apc_main"))
save_stage("11_APC_EEXY_QUERY", df_lot_hist_eexy, existing_parquet=snapshot_path("apc_eexy"))
save_stage("11_APC_MATCHED", df_lot_hist_apc)


## 12. 기적용 PSM 계수·shot 매칭 테이블
조회 결과와 LOT/chuck/shot 좌표가 연결된 df_psm_input을 각각 확인합니다. 조회한 shot 중심 좌표는 mm에서 µm로 변환합니다.


In [ ]:
def shot_sql(cfg, history):
    ids = history["apc_trocs_hist_index_no"].dropna().unique().tolist()
    columns = ",".join(PSM_COLUMNS)  # PSM 조회는 기존 SQL을 유지하고 TROCS는 16에서 따로 조회합니다.
    return f"""
SELECT CAST(apc_hist_index_no AS BIGINT) AS apc_hist_index_no,pos_x_valn,pos_y_valn,{columns}
FROM ees_ds_eai.apc_apc_photo_shot_input_hist
WHERE db_user = {quote(cfg.db_user_apc)} AND impala_insert_time >= now() - interval {cfg.days + 10} days
  {in_filter('CAST(apc_hist_index_no AS BIGINT)', ids, True)}
"""

def prepare_psm_input(shot, history, lots):
    if shot.empty or history.empty:
        return pd.DataFrame(columns=PSM_META + PSM_COLUMNS + ["match_key"])
    require_columns(shot, ["apc_hist_index_no", "pos_x_valn", "pos_y_valn"] + PSM_COLUMNS, "PSM INPUT")
    shot, history = shot.copy(), history.copy()
    numeric(shot, ["pos_x_valn", "pos_y_valn"] + PSM_COLUMNS)
    shot["apc_hist_index_no"] = integer(shot["apc_hist_index_no"])
    for col in ["apc_hist_index_no", "apc_trocs_hist_index_no"]:
        history[col] = integer(history[col])
    for col in ["event_tsdt", "tkin_tsdt"]:
        history[col] = pd.to_datetime(history[col], errors="coerce")
    subset = history[HISTORY_COLUMNS].dropna(subset=["apc_trocs_hist_index_no"]).sort_values(["event_tsdt", "tkin_tsdt"]).drop_duplicates("apc_trocs_hist_index_no", keep="last")
    if subset.empty:
        return pd.DataFrame(columns=PSM_META + PSM_COLUMNS + ["match_key"])
    joined = shot.merge(subset, left_on="apc_hist_index_no", right_on="apc_trocs_hist_index_no", how="left", validate="m:1", suffixes=("_shot", "_lot"))
    joined = joined.rename(columns={"apc_hist_index_no_lot": "apc_hist_index_no"})
    df = joined[PSM_META + PSM_COLUMNS].copy()
    df["pos_x_valn"], df["pos_y_valn"] = df["pos_x_valn"] * 1000, df["pos_y_valn"] * 1000
    df["lot_id"], df["sub_eqp_id"] = df["lot_id"].astype(str).str.strip(), df["sub_eqp_id"].astype(str).str.strip()
    keys = pd.DataFrame({"lot_id": lots["lotid"].astype(str), "sub_eqp_id": chuck_for_slot(lots["slotid"])})
    df = df.merge(keys.dropna().drop_duplicates(), on=["lot_id", "sub_eqp_id"], how="inner")
    df["fcp_x_round"], df["fcp_y_round"] = df["pos_x_valn"].round(2), df["pos_y_valn"].round(2)
    df["match_key"] = psm_key(df["lot_id"], df["sub_eqp_id"], df["fcp_x_round"], df["fcp_y_round"])
    return df

has_shot_ids = not df_lot_hist_apc.empty and df_lot_hist_apc["apc_trocs_hist_index_no"].notna().any()
sql_shot_input = shot_sql(cfg, df_lot_hist_apc) if has_shot_ids else None
df_shot_input_apc = fetch_frame("psm_shot", sql_shot_input) if sql_shot_input else pd.DataFrame()
df_psm_input = prepare_psm_input(df_shot_input_apc, df_lot_hist_apc, df_lotinfo_adi)
save_stage("12_PSM_SHOT_QUERY", df_shot_input_apc, existing_parquet=snapshot_path("psm_shot"))
save_stage("12_PSM_INPUT", df_psm_input)


## 13. 측정점에서 기적용 PSM 값 평가
이 셀은 기존 계수를 측정점에 입히는 부분입니다. raw 계산은 다음 셀에서 따로 실행합니다. PSM 키·계수 NaN·좌표 처리 규칙은 이 함수 안에서 수정할 수 있습니다.


In [ ]:
def cpe_k_to_fit(rx, ry):
    X_dx = np.vstack([
        np.ones(len(rx)),
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
        (rx ** 4) / 1e19, (rx ** 3 * ry) / 1e19, (rx ** 2 * ry ** 2) / 1e19, (rx * ry ** 3) / 1e19, (ry ** 4) / 1e19,
        (rx ** 5) / 1e23, (rx ** 4 * ry) / 1e23, (rx ** 3 * ry ** 2) / 1e23, (rx ** 2 * ry ** 3) / 1e23, (rx * ry ** 4) / 1e23, (ry ** 5) / 1e23,
        (rx ** 6) / 1e27, (rx ** 5 * ry) / 1e27, (rx ** 4 * ry ** 2) / 1e27, (rx ** 3 * ry ** 3) / 1e27, (rx ** 2 * ry ** 4) / 1e27, (rx * ry ** 5) / 1e27, (ry ** 6) / 1e27,
        (rx ** 7) / 1e31, (rx ** 6 * ry) / 1e31, (rx ** 5 * ry ** 2) / 1e31, (rx ** 4 * ry ** 3) / 1e31, (rx ** 3 * ry ** 4) / 1e31, (rx ** 2 * ry ** 5) / 1e31, (rx * ry ** 6) / 1e31, (ry ** 7) / 1e31
    ]).T

    X_dy = np.vstack([
        np.ones(len(ry)),
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12, (rx ** 3) / 1e12,
        (ry ** 4) / 1e19, (ry ** 3 * rx) / 1e19, (ry ** 2 * rx ** 2) / 1e19, (ry * rx ** 3) / 1e19, (rx ** 4) / 1e19,
        (ry ** 5) / 1e23, (ry ** 4 * rx) / 1e23, (ry ** 3 * rx ** 2) / 1e23, (ry ** 2 * rx ** 3) / 1e23, (ry * rx ** 4) / 1e23, (rx ** 5) / 1e23,
        (ry ** 6) / 1e27, (ry ** 5 * rx) / 1e27, (ry ** 4 * rx ** 2) / 1e27, (ry ** 3 * rx ** 3) / 1e27, (ry ** 2 * rx ** 4) / 1e27, (ry * rx ** 5) / 1e27, (rx ** 6) / 1e27,
        (ry ** 7) / 1e31, (ry ** 6 * rx) / 1e31, (ry ** 5 * rx ** 2) / 1e31, (ry ** 4 * rx ** 3) / 1e31, (ry ** 3 * rx ** 4) / 1e31, (ry ** 2 * rx ** 5) / 1e31, (ry * rx ** 6) / 1e31, (rx ** 7) / 1e31
    ]).T

    return X_dx, X_dy

def fit_applied_psm(frame, psm_input):
    df = frame.copy()
    df["psm_fit_x"], df["psm_fit_y"], df["psm_fit_status"] = np.nan, np.nan, "missing_psm"
    if not df.empty:
        require_columns(df, ["LOT_ID", "Wafer", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y", "X_reg_demrc", "Y_reg_demrc"], "PSM 적용")
        df["_psm_lot_key"], df["_psm_slot_key"] = df["LOT_ID"].astype(str).str.strip(), integer(df["Wafer"])
        df["sub_eqp_id"] = chuck_for_slot(df["Wafer"])
        df["fcp_x_round"], df["fcp_y_round"] = pd.to_numeric(df["fcp_x"], errors="coerce").round(2), pd.to_numeric(df["fcp_y"], errors="coerce").round(2)
        df["match_key"] = psm_key(df["_psm_lot_key"], df["sub_eqp_id"], df["fcp_x_round"], df["fcp_y_round"])
        if not psm_input.empty:
            require_columns(psm_input, ["match_key"] + PSM_COLUMNS, "PSM 계수")
            source = psm_input.copy()
            sort_cols = [col for col in ["event_tsdt", "tkin_tsdt"] if col in source]
            for col in sort_cols:
                source[col] = pd.to_datetime(source[col], errors="coerce")
            source = source.sort_values(sort_cols) if sort_cols else source
            source = source.dropna(subset=["match_key"]).drop_duplicates("match_key", keep="last")
            coefficients = np.ascontiguousarray(numeric(source, PSM_COLUMNS)[PSM_COLUMNS].to_numpy(float))
            lookup = {key: i for i, key in enumerate(source["match_key"])}
            rx, ry = pd.to_numeric(df["coordinate_X"], errors="coerce").to_numpy(float), pd.to_numeric(df["coordinate_Y"], errors="coerce").to_numpy(float)
            fit_x, fit_y, status = np.full(len(df), np.nan), np.full(len(df), np.nan), df["psm_fit_status"].to_numpy(object)
            for key, positions in df.groupby("match_key", sort=False, dropna=False).indices.items():
                idx = lookup.get(key)
                if idx is None:
                    continue
                values = coefficients[idx]
                if np.isnan(values).all():
                    status[positions] = "invalid_psm_k"
                    continue
                if np.isnan(rx[positions]).any() or np.isnan(ry[positions]).any():
                    status[positions] = "invalid_psm_coordinates"
                    continue
                # 원본 010.py: 부분 NaN 계수는 0, 모든 계수가 NaN이면 해당 shot을 건너뜁니다.
                values = np.nan_to_num(values, nan=0.0)
                mx, my = cpe_k_to_fit(rx[positions], ry[positions])
                fit_x[positions], fit_y[positions] = mx.dot(values[::2]), my.dot(values[1::2])
                status[positions] = "matched"
            df["psm_fit_x"], df["psm_fit_y"], df["psm_fit_status"] = fit_x, fit_y, status
    return df

df_adi_psm_fit = fit_applied_psm(df_adi_demrc, df_psm_input)
df_oco_psm_fit = fit_applied_psm(df_oco_demrc, df_psm_input)
psm_preview = ["LOT_ID", "Wafer", "TEST", "sub_eqp_id", "fcp_x", "fcp_y", "PSM_X", "PSM_Y", "psm_fit_x", "psm_fit_y", "psm_fit_status"]
save_stage("13_ADI_PSM_FIT", df_adi_psm_fit, columns=psm_preview)
save_stage("13_OCO_PSM_FIT", df_oco_psm_fit, columns=psm_preview)


## 14. PSM decorrect → raw 계산
ADI와 OCO의 raw 계산식을 여기서 직접 확인·수정합니다. 매칭 누락으로 계산할 수 없는 값은 NaN으로 유지합니다. 이 결과에는 마지막 MMO_MRC_EQP 필터가 아직 적용되지 않았습니다.
ADI는 `raw = Reg - mrc_fit - 실제 PERSHOT`, OCO는 `raw = Reg - mrc_fit - PSM 계수 평가값`입니다. 두 데이터의 PSM 출처가 다르므로 구분합니다. ADI raw가 실제 오차 기준이라는 정의는 이 생성식의 부호를 변경하라는 뜻이 아닙니다.


In [ ]:
df_adi_raw = df_adi_psm_fit.copy()
df_oco_raw = df_oco_psm_fit.copy()
for frame in [df_adi_raw, df_oco_raw]:
    for col in ["PSM_X", "PSM_Y"]:
        if col not in frame:
            frame[col] = np.nan
    numeric(frame, ["PSM_X", "PSM_Y"])

# ADI: 실제 PERSHOT 보정값을 되돌림
df_adi_raw["raw_x"] = df_adi_raw["X_reg_demrc"] - df_adi_raw["PSM_X"]
df_adi_raw["raw_y"] = df_adi_raw["Y_reg_demrc"] - df_adi_raw["PSM_Y"]
df_adi_raw["psm_source"] = "PERSHOT"

# OCO: 기적용 PSM 계수로 계산한 보정값을 되돌림
df_oco_raw["raw_x"] = df_oco_raw["X_reg_demrc"] - df_oco_raw["psm_fit_x"]
df_oco_raw["raw_y"] = df_oco_raw["Y_reg_demrc"] - df_oco_raw["psm_fit_y"]
df_oco_raw["psm_source"] = "APC_PSM_FIT"
for frame in [df_adi_raw, df_oco_raw]:
    frame["decorrect_valid"] = np.isfinite(frame[["raw_x", "raw_y"]].to_numpy(float)).all(axis=1)
raw_preview = ["LOT_ID", "Wafer", "TEST", "X_reg", "Y_reg", "X_reg_demrc", "Y_reg_demrc", "PSM_X", "PSM_Y", "psm_fit_x", "psm_fit_y", "raw_x", "raw_y", "decorrect_valid"]
save_stage("14_ADI_RAW_BEFORE_FILTER", df_adi_raw, columns=raw_preview)
save_stage("14_OCO_RAW_BEFORE_FILTER", df_oco_raw, columns=raw_preview)


## 15. 원본의 마지막 필터·wafer 요약·최종 결과
MMO_MRC_EQP가 없는 행만 원본처럼 제외합니다. 보정 누락 행을 추가로 제거하지 않습니다. df_final_adi/oco, df_wafer_summary와 df_saved_files가 최종 확인 변수입니다.


In [ ]:
def summarize(frame, mode):
    if frame.empty:
        return pd.DataFrame(columns=["mode", "LOT_ID", "Wafer", "rows", "mrc_matched", "psm_fit_matched", "valid_raw", "invalid_raw"])
    work = frame[["LOT_ID", "Wafer", "decorrect_valid", "mrc_status", "psm_fit_status"]].copy()
    work["mrc_matched"], work["psm_fit_matched"] = work["mrc_status"].eq("matched"), work["psm_fit_status"].eq("matched")
    out = work.groupby(["LOT_ID", "Wafer"], dropna=False).agg(rows=("decorrect_valid", "size"), mrc_matched=("mrc_matched", "sum"), psm_fit_matched=("psm_fit_matched", "sum"), valid_raw=("decorrect_valid", "sum")).reset_index()
    out.insert(0, "mode", mode)
    out["invalid_raw"] = out["rows"] - out["valid_raw"]
    return out

df_final_adi = df_adi_raw[df_adi_raw["MMO_MRC_EQP"].notna() & df_adi_raw["MMO_MRC_EQP"].astype(str).str.strip().ne("")].copy()
df_final_oco = df_oco_raw[df_oco_raw["MMO_MRC_EQP"].notna() & df_oco_raw["MMO_MRC_EQP"].astype(str).str.strip().ne("")].copy()
df_wafer_summary = pd.concat([summarize(df_final_adi, "ADI"), summarize(df_final_oco, "OCO")], ignore_index=True)
save_stage("15_ADI_FINAL", df_final_adi, columns=raw_preview)
save_stage("15_OCO_FINAL", df_final_oco, columns=raw_preview)
save_stage("15_WAFER_SUMMARY", df_wafer_summary)
df_saved_files = pd.DataFrame(stage_files.values())
display(df_saved_files)
display(FileLink(os.path.relpath(OUTPUT_DIR / "saved_files.csv", Path.cwd()), result_html_prefix="전체 중간 파일 목록: "))
source.save_manifest(cfg)
print("MRC/PSM decorrect까지 완료했습니다.")


## 16. TROCS RK1~72 조회 및 APC 이력 연결
11에서 선택한 EEXY/TROCS 이력 번호로 shot input을 조회하고, **main APC 번호**로 변환해 실측 데이터와 연결합니다. `pos_x/y_valn`은 DB의 mm를 µm로 한 번만 변환합니다.

과거 스냅샷으로 15까지 실행했다면 `TROCS_QUERY_FILE`에 **아래 SQL 조회 결과 원본 파일**(Parquet/Excel/CSV)을 지정할 수 있습니다. 파일은 `apc_hist_index_no`(TROCS 이력 번호), `pos_x/y_valn`(mm), `rk1~72_valn`을 포함해야 합니다. 가공된 µm 입력 파일을 지정하지 마세요. 파일 지정 없이 replay하려면 `trocs_shot` 스냅샷이 필요합니다.


In [ ]:
TROCS_QUERY_FILE = None  # 예: Path("trocs_shot_query.parquet"); None이면 live/replay 조회
TROCS_COLUMNS = [f"rk{i}_valn" for i in range(1, 73)]
TROCS_INPUT_COLUMNS = PSM_META + ["apc_trocs_hist_index_no"] + TROCS_COLUMNS
TROCS_WAFER_RADIUS = cfg.wafer_radius_um  # 원본 011.py: 147000 µm

def trocs_sql(cfg, history):
    ids = history["apc_trocs_hist_index_no"].dropna().unique().tolist()
    columns = ",".join(TROCS_COLUMNS)
    return f"""
SELECT CAST(apc_hist_index_no AS BIGINT) AS apc_hist_index_no,pos_x_valn,pos_y_valn,{columns}
FROM ees_ds_eai.apc_apc_photo_shot_input_hist
WHERE db_user = {quote(cfg.db_user_apc)} AND impala_insert_time >= now() - interval {cfg.days + 10} days
  {in_filter('CAST(apc_hist_index_no AS BIGINT)', ids, True)}
"""

def prepare_trocs_input(shot, history):
    if shot.empty or history.empty:
        return pd.DataFrame(columns=TROCS_INPUT_COLUMNS + ["trocs_key"])
    require_columns(shot, ["apc_hist_index_no", "pos_x_valn", "pos_y_valn"] + TROCS_COLUMNS, "TROCS QUERY")
    shot, history = shot.copy(), history.copy()
    numeric(shot, ["pos_x_valn", "pos_y_valn"] + TROCS_COLUMNS)
    shot["apc_hist_index_no"] = integer(shot["apc_hist_index_no"])
    for col in ["apc_hist_index_no", "apc_trocs_hist_index_no"]:
        history[col] = integer(history[col])
    for col in ["event_tsdt", "tkin_tsdt"]:
        history[col] = pd.to_datetime(history[col], errors="coerce")
    history = history[HISTORY_COLUMNS].dropna(subset=["apc_trocs_hist_index_no"])
    history = history.sort_values(["event_tsdt", "tkin_tsdt"]).drop_duplicates("apc_trocs_hist_index_no", keep="last")
    joined = shot.merge(history, left_on="apc_hist_index_no", right_on="apc_trocs_hist_index_no", how="left", validate="m:1", suffixes=("_shot", "_lot"))
    joined = joined.rename(columns={"apc_hist_index_no_lot": "apc_hist_index_no"})
    out = joined[TROCS_INPUT_COLUMNS].copy()
    out["pos_x_valn"], out["pos_y_valn"] = out["pos_x_valn"] * 1000, out["pos_y_valn"] * 1000
    out["trocs_key"] = trocs_key(out["apc_hist_index_no"], out["pos_x_valn"], out["pos_y_valn"])
    return out.sort_values(["event_tsdt", "tkin_tsdt"]).dropna(subset=["trocs_key"]).drop_duplicates("trocs_key", keep="last").reset_index(drop=True)

def trocs_key(apc, x, y):
    key = integer(apc).astype("string") + "_" + coord_key(x) + "_" + coord_key(y)
    return key.mask(pd.to_numeric(x, errors="coerce").isna() | pd.to_numeric(y, errors="coerce").isna())

sql_trocs_input = trocs_sql(cfg, df_lot_hist_apc) if has_shot_ids else None
if TROCS_QUERY_FILE is not None:
    path = Path(TROCS_QUERY_FILE)
    readers = {".parquet": pd.read_parquet, ".xlsx": pd.read_excel, ".csv": pd.read_csv}
    if path.suffix.lower() not in readers:
        raise ValueError("TROCS_QUERY_FILE은 .parquet/.xlsx/.csv 파일을 지정하세요.")
    df_trocs_shot_query = readers[path.suffix.lower()](path)
elif sql_trocs_input:
    df_trocs_shot_query = fetch_frame("trocs_shot", sql_trocs_input)
else:
    df_trocs_shot_query = pd.DataFrame(columns=["apc_hist_index_no", "pos_x_valn", "pos_y_valn"] + TROCS_COLUMNS)
df_trocs_input = prepare_trocs_input(df_trocs_shot_query, df_lot_hist_apc)
save_stage("16_TROCS_QUERY", df_trocs_shot_query)
save_stage("16_TROCS_INPUT_UM", df_trocs_input)


## 17. TROCS wafer 전체 mark 계산 함수
`011.py`처럼 `UNIQUE_ID`별 실제 TEST mark 좌표를 전체 shot 배치에 펼칩니다. shot 후보는 원본의 -10~10 범위, 네 모서리 판정 및 mark 반경 `<147000 µm`를 유지하며 배열 계산으로 생성합니다. 동일 TEST의 첫 유효 좌표를 사용합니다.

TROCS는 13에서 정의한 **72계수 행렬**로 계산합니다. 일부 계수 결측은 원본처럼 0으로 계산하고, 전체 결측/미연결 shot은 NaN 및 상태를 남깁니다. 원본과 같이 보정값이 없는 shot은 `ALLMARKS` 결과에서 제외하며, `SHOT_DIAGNOSTICS`에는 보존합니다.


In [ ]:
TROCS_META = ["STEPSEQ", "LOT_ID", "Wafer", "GROUP", "P_EQPID", "Photo_PPID", "P_TIME", "M_TIME", "ChuckID", "ReticleID", "Base_EQP1", "MMO_MRC_EQP", "M_STEPSEQ", "STEP_PITCH_X", "STEP_PITCH_Y", "MAP_SHIFT_X", "MAP_SHIFT_Y", "photo_transn_seq", "apc_trocs_hist_index_no"]
TROCS_MARK_COLUMNS = ["Test_ID", "Shot_X", "Shot_Y", "Coordinate_X", "Coordinate_Y", "Wafer_X", "Wafer_Y", "Radius", "Inside_Radius", "fcp_x", "fcp_y", "apc_hist_index_no", "UNIQUE_ID"] + TROCS_META + ["trocs_fit_all_x", "trocs_fit_all_y", "trocs_status"]

def build_all_trocs_marks(test_inf, layout, radius):
    pitch_x, pitch_y, shift_x, shift_y = layout
    sx, sy = np.meshgrid(np.arange(-10, 11), np.arange(-10, 11), indexing="ij")
    sx, sy = sx.ravel(), sy.ravel()
    cx, cy = sx * pitch_x + shift_x, sy * pitch_y + shift_y
    dx, dy = np.array([-1, -1, 1, 1]) * pitch_x / 2, np.array([-1, 1, 1, -1]) * pitch_y / 2
    corners = np.sqrt((cx[:, None] + dx) ** 2 + (cy[:, None] + dy) ** 2)
    keep = corners.min(axis=1) <= radius
    sx, sy, cx, cy = sx[keep], sy[keep], cx[keep], cy[keep]
    n = len(test_inf)
    rx, ry = np.tile(test_inf[:, 0], len(sx)), np.tile(test_inf[:, 1], len(sy))
    wx, wy = np.repeat(cx, n) + rx, np.repeat(cy, n) + ry
    r = np.sqrt(wx ** 2 + wy ** 2)
    out = pd.DataFrame({"Test_ID": np.tile(np.arange(1, n + 1), len(sx)), "Shot_X": np.repeat(sx, n), "Shot_Y": np.repeat(sy, n), "Coordinate_X": rx, "Coordinate_Y": ry, "Wafer_X": wx, "Wafer_Y": wy, "Radius": r, "Inside_Radius": r < radius})
    return out.loc[out["Inside_Radius"]].reset_index(drop=True)

def calculate_trocs_allmarks(frame, trocs_input, radius=TROCS_WAFER_RADIUS):
    required = ["UNIQUE_ID", "apc_hist_index_no", "STEP_PITCH_X", "STEP_PITCH_Y", "MAP_SHIFT_X", "MAP_SHIFT_Y", "coordinate_X", "coordinate_Y"]
    require_columns(frame, required, "TROCS MEASURED")
    lookup = trocs_input.dropna(subset=["trocs_key"]).drop_duplicates("trocs_key", keep="last").set_index("trocs_key")
    blocks, diagnostics = [], []
    for uid, grp in frame.groupby("UNIQUE_ID"):
        layout = pd.to_numeric(grp[["STEP_PITCH_X", "STEP_PITCH_Y", "MAP_SHIFT_X", "MAP_SHIFT_Y"]].iloc[0], errors="coerce").to_numpy(dtype=float)
        apcs = integer(grp["apc_hist_index_no"]).dropna().unique()
        reason = "invalid_layout" if not np.isfinite(layout).all() or (layout[:2] <= 0).any() else "missing_apc" if not len(apcs) else None
        meta = {col: grp[col].dropna().iloc[0] if grp[col].notna().any() else None for col in TROCS_META if col in grp}
        if reason:
            diagnostics.append(dict(meta, UNIQUE_ID=uid, trocs_status=reason, n_generated=0, n_fitted=0))
            continue
        if len(apcs) > 1:
            warnings.warn(f"{uid}: APC 번호가 {len(apcs)}개입니다. 원본처럼 첫 번호 {apcs[0]} 사용.")
        coords = numeric(grp.copy(), ["coordinate_X", "coordinate_Y"])
        if "TEST" in coords:
            coords = coords.dropna(subset=["TEST", "coordinate_X", "coordinate_Y"]).drop_duplicates("TEST", keep="first").sort_values("TEST")
        else:
            coords = coords.dropna(subset=["coordinate_X", "coordinate_Y"]).drop_duplicates(["coordinate_X", "coordinate_Y"])
        test_inf = coords[["coordinate_X", "coordinate_Y"]].to_numpy(dtype=float)
        if not len(test_inf):
            diagnostics.append(dict(meta, UNIQUE_ID=uid, trocs_status="no_valid_marks", n_generated=0, n_fitted=0))
            continue
        marks = build_all_trocs_marks(test_inf, layout, radius)
        marks["fcp_x"], marks["fcp_y"] = marks["Shot_X"] * layout[0] + layout[2], marks["Shot_Y"] * layout[1] + layout[3]
        marks["apc_hist_index_no"], marks["UNIQUE_ID"] = int(apcs[0]), uid
        for col, value in meta.items():
            marks[col] = value
        marks["_trocs_key"] = trocs_key(marks["apc_hist_index_no"], marks["fcp_x"], marks["fcp_y"])
        marks["trocs_fit_all_x"], marks["trocs_fit_all_y"], marks["trocs_status"] = np.nan, np.nan, "missing_coefficients"
        for key, shot in marks.groupby("_trocs_key"):
            status, partial = "missing_coefficients", False
            if key in lookup.index:
                coeff = lookup.loc[key, TROCS_COLUMNS].to_numpy(dtype=float)
                if np.isnan(coeff).all():
                    status = "all_coefficients_nan"
                else:
                    partial = bool(np.isnan(coeff).any())
                    coeff = np.nan_to_num(coeff, nan=0.0)
                    mx, my = cpe_k_to_fit(shot["Coordinate_X"].to_numpy(dtype=float), shot["Coordinate_Y"].to_numpy(dtype=float))
                    marks.loc[shot.index, "trocs_fit_all_x"] = mx @ coeff[::2]
                    marks.loc[shot.index, "trocs_fit_all_y"] = my @ coeff[1::2]
                    status = "partial_coefficients_zero_filled" if partial else "matched"
            marks.loc[shot.index, "trocs_status"] = status
            n_fit = int(marks.loc[shot.index, ["trocs_fit_all_x", "trocs_fit_all_y"]].notna().any(axis=1).sum())
            row = dict(meta, UNIQUE_ID=uid, apc_hist_index_no=int(apcs[0]), fcp_x=shot["fcp_x"].iloc[0], fcp_y=shot["fcp_y"].iloc[0], Shot_X=int(shot["Shot_X"].iloc[0]), Shot_Y=int(shot["Shot_Y"].iloc[0]), trocs_status=status, n_generated=len(shot), n_fitted=n_fit)
            diagnostics.append(row)
        valid = marks[["trocs_fit_all_x", "trocs_fit_all_y"]].notna().any(axis=1).groupby([marks["Shot_X"], marks["Shot_Y"]]).transform("any")
        blocks.append(marks.loc[valid].drop(columns="_trocs_key"))
    allmarks = pd.concat(blocks, ignore_index=True) if blocks else pd.DataFrame(columns=TROCS_MARK_COLUMNS)
    diag_columns = TROCS_META + ["UNIQUE_ID", "apc_hist_index_no", "fcp_x", "fcp_y", "Shot_X", "Shot_Y", "trocs_status", "n_generated", "n_fitted"]
    diag = pd.DataFrame(diagnostics).reindex(columns=diag_columns)
    return allmarks, diag


## 18. TROCS 계산 실행 및 wafer/shot 자료 저장
`ALLMARKS`는 wafer 전체 mark별 보정값, `SHOT_DATA`는 shot별 보정값 통계, `SHOT_COEFFICIENTS`는 실제 사용한 RK1~72, `WAFER_DATA`는 wafer별 통계입니다. `SHOT_DIAGNOSTICS`에는 계수 미연결·전체 결측·잘못된 배치도 기록합니다. RMS는 `sqrt(mean(value²))`이며, 여기서는 TROCS 보정값 크기의 통계입니다.


In [ ]:
WAFER_KEYS = ["LOT_ID", "Wafer"]
SHOT_KEYS = WAFER_KEYS + ["fcp_x", "fcp_y"]
POINT_KEYS = ["UNIQUE_ID", "LOT_ID", "Wafer", "GROUP", "TEST", "DieX", "DieY", "apc_hist_index_no", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"]

def summarize_values(frame, keys, value_cols):
    columns = keys + ["n_points"] + [f"{col}_{stat}" for col in value_cols for stat in ["n_valid", "mean", "std", "rms", "max_abs"]]
    if frame.empty:
        return pd.DataFrame(columns=columns)
    grouped = frame.groupby(keys, dropna=False)
    result = grouped.size().rename("n_points").to_frame()
    for col in value_cols:
        work = frame[keys].copy()
        work["_value"] = pd.to_numeric(frame[col], errors="coerce")
        work["_square"], work["_abs"] = work["_value"] ** 2, work["_value"].abs()
        stats = work.groupby(keys, dropna=False).agg(n_valid=("_value", "count"), mean=("_value", "mean"), std=("_value", "std"), mean_square=("_square", "mean"), max_abs=("_abs", "max"))
        stats["rms"] = np.sqrt(stats.pop("mean_square"))
        result = result.join(stats.add_prefix(col + "_"))
    return result.reset_index().reindex(columns=columns)

def trocs_shot_coefficients(allmarks, trocs_input):
    keys = [col for col in ["UNIQUE_ID", "LOT_ID", "Wafer", "GROUP", "apc_hist_index_no", "fcp_x", "fcp_y", "Shot_X", "Shot_Y", "trocs_status"] if col in allmarks]
    shots = allmarks[keys].drop_duplicates().copy()
    shots["trocs_key"] = trocs_key(shots["apc_hist_index_no"], shots["fcp_x"], shots["fcp_y"])
    payload = ["trocs_key", "event_tsdt", "tkin_tsdt", "apc_trocs_hist_index_no"] + TROCS_COLUMNS
    return shots.merge(trocs_input[payload], on="trocs_key", how="left", validate="m:1")

df_trocs_allmarks_adi, df_trocs_diagnostics_adi = calculate_trocs_allmarks(df_final_adi, df_trocs_input, TROCS_WAFER_RADIUS)
df_trocs_allmarks_oco, df_trocs_diagnostics_oco = calculate_trocs_allmarks(df_final_oco, df_trocs_input, TROCS_WAFER_RADIUS)
df_trocs_shot_coeff_adi = trocs_shot_coefficients(df_trocs_allmarks_adi, df_trocs_input)
df_trocs_shot_coeff_oco = trocs_shot_coefficients(df_trocs_allmarks_oco, df_trocs_input)
df_trocs_wafer_adi = summarize_values(df_trocs_allmarks_adi, WAFER_KEYS, ["trocs_fit_all_x", "trocs_fit_all_y"])
df_trocs_wafer_oco = summarize_values(df_trocs_allmarks_oco, WAFER_KEYS, ["trocs_fit_all_x", "trocs_fit_all_y"])
df_trocs_shot_adi = summarize_values(df_trocs_allmarks_adi, SHOT_KEYS, ["trocs_fit_all_x", "trocs_fit_all_y"])
df_trocs_shot_oco = summarize_values(df_trocs_allmarks_oco, SHOT_KEYS, ["trocs_fit_all_x", "trocs_fit_all_y"])
for mode, marks, diagnostics, coeff, wafer, shot in [("ADI", df_trocs_allmarks_adi, df_trocs_diagnostics_adi, df_trocs_shot_coeff_adi, df_trocs_wafer_adi, df_trocs_shot_adi), ("OCO", df_trocs_allmarks_oco, df_trocs_diagnostics_oco, df_trocs_shot_coeff_oco, df_trocs_wafer_oco, df_trocs_shot_oco)]:
    save_stage(f"18_{mode}_TROCS_ALLMARKS", marks)
    save_stage(f"18_{mode}_TROCS_SHOT_DIAGNOSTICS", diagnostics)
    save_stage(f"18_{mode}_TROCS_SHOT_COEFFICIENTS", coeff)
    save_stage(f"18_{mode}_TROCS_WAFER_DATA", wafer)
    save_stage(f"18_{mode}_TROCS_SHOT_DATA", shot)


## 19. 실측 point에 TROCS 보정값 연결
원본 `012.py`와 동일하게 **APC 번호 + fcp_x/y + local coordinate_X/Y**를 소수점 2자리로 맞춰 연결합니다. PSM의 LOT/chuck 기반 `match_key`는 그대로 보관합니다. 실측 행을 삭제하지 않으며 미연결 TROCS는 NaN으로 남깁니다.

`df_adi_trocs`, `df_oco_trocs`에 `trocs_fit_x/y`가 추가됩니다. `raw_x/y`는 15의 MRC/PSM decorrect 값 그대로입니다. 이후 회귀 입력에 TROCS를 차감하려면 이 단계부터 별도 검토가 필요합니다.


In [ ]:
def attach_trocs(frame, allmarks):
    out = frame.drop(columns=["trocs_fit_x", "trocs_fit_y", "trocs_status"], errors="ignore").copy().reset_index(drop=True)
    out["apc_hist_index_no"] = integer(out["apc_hist_index_no"])
    if allmarks.empty:
        out["trocs_fit_x"], out["trocs_fit_y"], out["trocs_status"] = np.nan, np.nan, "unmatched"
        return out
    def point_key(df, x_col, y_col):
        key = trocs_key(df["apc_hist_index_no"], df["fcp_x"], df["fcp_y"]) + "_" + coord_key(df[x_col]) + "_" + coord_key(df[y_col])
        return key.mask(pd.to_numeric(df[x_col], errors="coerce").isna() | pd.to_numeric(df[y_col], errors="coerce").isna())
    out["_trocs_point_key"] = point_key(out, "coordinate_X", "coordinate_Y")
    payload = allmarks[["apc_hist_index_no", "fcp_x", "fcp_y", "Coordinate_X", "Coordinate_Y", "trocs_fit_all_x", "trocs_fit_all_y", "trocs_status"]].copy()
    payload["_trocs_point_key"] = point_key(payload, "Coordinate_X", "Coordinate_Y")
    payload = payload.dropna(subset=["_trocs_point_key"]).drop_duplicates("_trocs_point_key", keep="last")
    payload = payload[["_trocs_point_key", "trocs_fit_all_x", "trocs_fit_all_y", "trocs_status"]].rename(columns={"trocs_fit_all_x": "trocs_fit_x", "trocs_fit_all_y": "trocs_fit_y"})
    out = out.merge(payload, on="_trocs_point_key", how="left", validate="m:1", sort=False).drop(columns="_trocs_point_key")
    out["trocs_status"] = out["trocs_status"].fillna("unmatched")
    return out

df_adi_trocs = attach_trocs(df_final_adi, df_trocs_allmarks_adi)
df_oco_trocs = attach_trocs(df_final_oco, df_trocs_allmarks_oco)
trocs_preview = POINT_KEYS + ["raw_x", "raw_y", "trocs_fit_x", "trocs_fit_y", "trocs_status"]
save_stage("19_ADI_TROCS_ATTACHED", df_adi_trocs, columns=trocs_preview)
save_stage("19_OCO_TROCS_ATTACHED", df_oco_trocs, columns=trocs_preview)


## 20. Wafer별 WK/RK 동시 회귀 함수
`012.py`와 같은 `(LOT_ID, Wafer)` 그룹 및 `osr_wk20p_rk20p` 행렬로 X/Y 각각 19개 계수를 **한 번에 최소제곱 회귀**합니다. 처음 10개는 WK, 뒤 9개는 RK입니다. RK에는 translation이 없고 RK3~20을 사용합니다.

원본과 같이 X/Y 및 좌표가 모두 유효한 점이 19개 미만이면 회귀를 건너뛰고 residual은 NaN으로 둡니다. `WK_RK_MIN_POINTS`로 기준을 수정할 수 있습니다. 충분한 점 수가 있어도 배치에 따라 행렬 rank는 부족할 수 있으므로 21에서 rank 진단을 별도 저장합니다.
**회귀 입력은 `raw_x/raw_y`입니다.** 뒤의 `df_adi_reg/df_oco_reg`는 raw 회귀 결과 테이블을 뜻하며, 계측기 `X_reg/Y_reg` 회귀 결과가 아닙니다.


In [ ]:
WK_RK_MIN_POINTS = 19

def osr_wk20p_rk20p(x, y, rx, ry):
    """
    회사 코드의 osr_wk20p_rk20p 구조를 그대로 사용.
    x, y   : wafer/field level 좌표 -> fcp_x, fcp_y
    rx, ry : shot 내부 좌표         -> coordinate_X, coordinate_Y
    반환:
        X_dx: X 방향 회귀용 design matrix (N x 19)
        X_dy: Y 방향 회귀용 design matrix (N x 19)
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    X_dx = np.vstack([
        np.ones(len(x)),
        x / 1e6, -y / 1e6,
        (x ** 2) / 1e12, (x * y) / 1e12, (y ** 2) / 1e12,
        (x ** 3) / 1e15, (x ** 2 * y) / 1e15, (x * y ** 2) / 1e15, (y ** 3) / 1e15,
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
    ]).T
    X_dy = np.vstack([
        np.ones(len(y)),
        y / 1e6, x / 1e6,
        (y ** 2) / 1e12, (y * x) / 1e12, (x ** 2) / 1e12,
        (y ** 3) / 1e15, (y ** 2 * x) / 1e15, (y * x ** 2) / 1e15, (x ** 3) / 1e15,
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12, (rx ** 3) / 1e12,
    ]).T
    return X_dx, X_dy
COEFF_KEYS_DX = [
    "WK1", "WK3", "WK5", "WK7", "WK9", "WK11", "WK13", "WK15", "WK17", "WK19",
    "RK3", "RK5", "RK7", "RK9", "RK11", "RK13", "RK15", "RK17", "RK19",
]
COEFF_KEYS_DY = [
    "WK2", "WK4", "WK6", "WK8", "WK10", "WK12", "WK14", "WK16", "WK18", "WK20",
    "RK4", "RK6", "RK8", "RK10", "RK12", "RK14", "RK16", "RK18", "RK20",
]
WK_COLS_DX = COEFF_KEYS_DX[:10]
RK_COLS_DX = COEFF_KEYS_DX[10:]
WK_COLS_DY = COEFF_KEYS_DY[:10]
RK_COLS_DY = COEFF_KEYS_DY[10:]

def normalize_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def fit_lstsq(X, y):
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ coef
    return coef, pred

def run_company_joint_regression(df, label="ADI", min_points=19):
    """
    wafer 단위로 raw_x/raw_y를 회사 룰 osr_wk20p_rk20p로 동시 회귀.
    입력:
      - raw_x, raw_y
      - fcp_x, fcp_y
      - coordinate_X, coordinate_Y
      - LOT_ID, Wafer
    출력:
      - wk_fit_x/y
      - rk_fit_x/y
      - total_fit_x/y
      - raw_minus_total_fit_x/y
      - wafer별 coefficient dataframe
    """
    df = df.copy()
    required_cols = [
        "LOT_ID", "Wafer",
        "raw_x", "raw_y",
        "fcp_x", "fcp_y",
        "coordinate_X", "coordinate_Y",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{label}] 필수 컬럼 누락: {missing}")
    normalize_numeric(df, [
        "raw_x", "raw_y",
        "fcp_x", "fcp_y",
        "coordinate_X", "coordinate_Y",
    ])
    result_cols = [
        "wk_fit_x", "wk_fit_y",
        "rk_fit_x", "rk_fit_y",
        "total_fit_x", "total_fit_y",
        "raw_minus_total_fit_x", "raw_minus_total_fit_y",
    ]
    for c in result_cols:
        df[c] = np.nan
    coef_rows = []
    summary_rows = []
    for (lot_id, wafer), grp in df.groupby(["LOT_ID", "Wafer"], dropna=False):
        idx = grp.index
        valid_mask = (
            grp["raw_x"].notna() &
            grp["raw_y"].notna() &
            grp["fcp_x"].notna() &
            grp["fcp_y"].notna() &
            grp["coordinate_X"].notna() &
            grp["coordinate_Y"].notna()
        )
        work = grp.loc[valid_mask].copy()
        n_valid = len(work)
        if n_valid < min_points:
            summary_rows.append({
                "LOT_ID": lot_id,
                "Wafer": wafer,
                "n_total": len(grp),
                "n_valid": n_valid,
                "status": "skip_too_few_points",
            })
            continue
        x = work["fcp_x"].to_numpy(dtype=float)
        y = work["fcp_y"].to_numpy(dtype=float)
        rx = work["coordinate_X"].to_numpy(dtype=float)
        ry = work["coordinate_Y"].to_numpy(dtype=float)
        raw_x = work["raw_x"].to_numpy(dtype=float)
        raw_y = work["raw_y"].to_numpy(dtype=float)
        X_dx, X_dy = osr_wk20p_rk20p(x, y, rx, ry)
        coef_x, pred_x = fit_lstsq(X_dx, raw_x)
        coef_y, pred_y = fit_lstsq(X_dy, raw_y)
        wk_fit_x = X_dx[:, :10] @ coef_x[:10]
        rk_fit_x = X_dx[:, 10:] @ coef_x[10:]
        wk_fit_y = X_dy[:, :10] @ coef_y[:10]
        rk_fit_y = X_dy[:, 10:] @ coef_y[10:]
        total_fit_x = wk_fit_x + rk_fit_x
        total_fit_y = wk_fit_y + rk_fit_y
        residual_x = raw_x - total_fit_x
        residual_y = raw_y - total_fit_y
        valid_idx = work.index
        df.loc[valid_idx, "wk_fit_x"] = wk_fit_x
        df.loc[valid_idx, "rk_fit_x"] = rk_fit_x
        df.loc[valid_idx, "total_fit_x"] = total_fit_x
        df.loc[valid_idx, "raw_minus_total_fit_x"] = residual_x
        df.loc[valid_idx, "wk_fit_y"] = wk_fit_y
        df.loc[valid_idx, "rk_fit_y"] = rk_fit_y
        df.loc[valid_idx, "total_fit_y"] = total_fit_y
        df.loc[valid_idx, "raw_minus_total_fit_y"] = residual_y
        coef_row = {
            "LOT_ID": lot_id,
            "Wafer": wafer,
            "n_total": len(grp),
            "n_valid": n_valid,
        }
        for name, val in zip(COEFF_KEYS_DX, coef_x):
            coef_row[f"{name}_X"] = val
        for name, val in zip(COEFF_KEYS_DY, coef_y):
            coef_row[f"{name}_Y"] = val
        coef_rows.append(coef_row)
        summary_rows.append({
            "LOT_ID": lot_id,
            "Wafer": wafer,
            "n_total": len(grp),
            "n_valid": n_valid,
            "rms_raw_x": float(np.sqrt(np.nanmean(raw_x ** 2))),
            "rms_raw_y": float(np.sqrt(np.nanmean(raw_y ** 2))),
            "rms_residual_x": float(np.sqrt(np.nanmean(residual_x ** 2))),
            "rms_residual_y": float(np.sqrt(np.nanmean(residual_y ** 2))),
            "status": "ok",
        })
    coef_df = pd.DataFrame(coef_rows)
    summary_df = pd.DataFrame(summary_rows)
    print(f"[{label}] regression 완료")
    print(f"  - wafer 수: {df[['LOT_ID','Wafer']].drop_duplicates().shape[0]}")
    print(f"  - coefficient rows: {len(coef_df)}")
    print(f"  - summary rows: {len(summary_df)}")
    return df, coef_df, summary_df


## 21. WK/RK 제거 실행 및 성분별 저장
`wk_fit_x/y`와 `rk_fit_x/y`는 실측 위치에서 제거한 값입니다. `raw_minus_total_fit_x/y = raw_x/y - wk_fit_x/y - rk_fit_x/y`가 CPE 입력입니다.

- `WK_WAFER_COEFFICIENTS`: WK1~20 계수만 분리. `RK_WAFER_COEFFICIENTS`: wafer 공통 RK3~20 계수만 분리.
- `WK_REMOVED_POINTS`, `RK_REMOVED_POINTS`: 실제 각 point에서 제거한 WK/RK 값.
- `WK_RK_WAFER_DATA`, `WK_RK_SHOT_DATA`: 제거 전후 point 통계. `WK_RK_WAFER_DIAGNOSTICS`: 원본 회귀 상태·RMS와 추가 rank 진단.


In [ ]:
def split_wk_rk_coefficients(coeff):
    base = ["LOT_ID", "Wafer", "n_total", "n_valid"]
    wk = base + [f"{key}_{axis}" for axis, names in [("X", WK_COLS_DX), ("Y", WK_COLS_DY)] for key in names]
    rk = base + [f"{key}_{axis}" for axis, names in [("X", RK_COLS_DX), ("Y", RK_COLS_DY)] for key in names]
    return coeff.reindex(columns=wk), coeff.reindex(columns=rk)

def wk_rk_rank_diagnostics(frame, summary):
    rank_columns = WAFER_KEYS + ["rank_x", "rank_y", "n_parameters_per_axis", "rank_deficient"]
    if frame.empty:
        return summary.reindex(columns=list(dict.fromkeys(list(summary.columns) + rank_columns)))
    rows = []
    cols = ["raw_x", "raw_y", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"]
    for key, grp in frame.groupby(WAFER_KEYS, dropna=False):
        valid = numeric(grp[cols].copy(), cols).dropna()
        if not valid.empty:
            mx, my = osr_wk20p_rk20p(*(valid[col].to_numpy(dtype=float) for col in cols[2:]))
            rank_x, rank_y = np.linalg.matrix_rank(mx), np.linalg.matrix_rank(my)
        else:
            rank_x, rank_y = 0, 0
        rows.append({"LOT_ID": key[0], "Wafer": key[1], "rank_x": rank_x, "rank_y": rank_y, "n_parameters_per_axis": 19, "rank_deficient": rank_x < 19 or rank_y < 19})
    rank = pd.DataFrame(rows, columns=rank_columns)
    return summary.reindex(columns=list(dict.fromkeys(WAFER_KEYS + list(summary.columns)))).merge(rank, on=WAFER_KEYS, how="left", validate="1:1")

df_adi_reg, coef_adi, summary_adi = run_company_joint_regression(df_adi_trocs, label="ADI", min_points=WK_RK_MIN_POINTS)
df_oco_reg, coef_oco, summary_oco = run_company_joint_regression(df_oco_trocs, label="OCO", min_points=WK_RK_MIN_POINTS)
df_wk_coeff_adi, df_rk_coeff_adi = split_wk_rk_coefficients(coef_adi)
df_wk_coeff_oco, df_rk_coeff_oco = split_wk_rk_coefficients(coef_oco)
df_wk_rk_diag_adi, df_wk_rk_diag_oco = wk_rk_rank_diagnostics(df_adi_reg, summary_adi), wk_rk_rank_diagnostics(df_oco_reg, summary_oco)
wk_rk_values = ["raw_x", "raw_y", "wk_fit_x", "wk_fit_y", "rk_fit_x", "rk_fit_y", "raw_minus_total_fit_x", "raw_minus_total_fit_y"]
df_wk_rk_wafer_adi, df_wk_rk_wafer_oco = summarize_values(df_adi_reg, WAFER_KEYS, wk_rk_values), summarize_values(df_oco_reg, WAFER_KEYS, wk_rk_values)
df_wk_rk_shot_adi, df_wk_rk_shot_oco = summarize_values(df_adi_reg, SHOT_KEYS, wk_rk_values), summarize_values(df_oco_reg, SHOT_KEYS, wk_rk_values)
df_wk_removed_adi = df_adi_reg[[col for col in POINT_KEYS + ["wk_fit_x", "wk_fit_y"] if col in df_adi_reg]].copy()
df_wk_removed_oco = df_oco_reg[[col for col in POINT_KEYS + ["wk_fit_x", "wk_fit_y"] if col in df_oco_reg]].copy()
df_rk_removed_adi = df_adi_reg[[col for col in POINT_KEYS + ["rk_fit_x", "rk_fit_y"] if col in df_adi_reg]].copy()
df_rk_removed_oco = df_oco_reg[[col for col in POINT_KEYS + ["rk_fit_x", "rk_fit_y"] if col in df_oco_reg]].copy()
wk_rk_results = [
    ("ADI", df_adi_reg, df_wk_coeff_adi, df_rk_coeff_adi, df_wk_removed_adi, df_rk_removed_adi, df_wk_rk_diag_adi, df_wk_rk_wafer_adi, df_wk_rk_shot_adi),
    ("OCO", df_oco_reg, df_wk_coeff_oco, df_rk_coeff_oco, df_wk_removed_oco, df_rk_removed_oco, df_wk_rk_diag_oco, df_wk_rk_wafer_oco, df_wk_rk_shot_oco),
]
for mode, reg, wk, rk, wk_points, rk_points, diag, wafer, shot in wk_rk_results:
    save_stage(f"21_{mode}_WK_RK_REMOVED", reg, columns=POINT_KEYS + wk_rk_values)
    save_stage(f"21_{mode}_WK_WAFER_COEFFICIENTS", wk)
    save_stage(f"21_{mode}_RK_WAFER_COEFFICIENTS", rk)
    save_stage(f"21_{mode}_WK_REMOVED_POINTS", wk_points)
    save_stage(f"21_{mode}_RK_REMOVED_POINTS", rk_points)
    save_stage(f"21_{mode}_WK_RK_WAFER_DIAGNOSTICS", diag)
    save_stage(f"21_{mode}_WK_RK_WAFER_DATA", wafer)
    save_stage(f"21_{mode}_WK_RK_SHOT_DATA", shot)


## 22. CPE38 설정, 행렬 및 span 제한
**현재 `013.py`의 계산 순서와 설정값을 유지하는 1차 구현입니다.** X21/Y17개 계수의 38-parameter 모델을 사용합니다. 이후 CPE 변경은 22~26에서 수정하고 27~28을 다시 실행하면 됩니다. CPE는 아래 `SPAN_MODE`, ridge/G-opt/virtual grid/box 옵션을 명시적으로 전달합니다.

주의할 해석: 현재 span은 virtual point의 각 term 기여값 `max-min`이고, RK1/2 상수 term은 span=0입니다. box 상한은 원본의 `max(abs(min), abs(max))` 기준입니다. `inner/partial` 판정도 원본의 유효 point 수와 virtual point 수 비교를 유지합니다. 남은 G-opt/span 초과는 27의 진단에 표시하며 결과 행을 삭제하지 않습니다.
CPE는 raw에서 wafer WK/RK를 제거한 `raw_minus_total_fit_x/y`를 입력으로 사용합니다.


In [ ]:
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

SPAN_MODE = "OVO3_38P_F2F"
RIDGE_REGRESSION = True
G_OPT_SPEC = 1.0
USE_VIRTUAL_CHIP_POINTS = True   # VBA virtual grid: chip별 4 corner + center point
G_OPT_TOLERANCE = 0.001          # VBA: G_opt_tolerlance = 0.001
SPAN_TOLERANCE = 0.01            # VBA: span_tolerlance = 0.01
MINIMUM_RIDGE = 0.001            # VBA: minimum_Ridge = 0.001
ADD_RIDGE = 10.0                 # VBA: add_ridge = 10
MAX_GOPT_RETURN = 50             # VBA: max_prevent_fallback_retrun = 50
MAX_SPAN_ITER = 5000             # VBA: max_iter = 5000
CD_TOL = 1e-6                    # VBA: tol = 0.000001
CPE_EVAL_GRID_N = 10
MIN_POINTS_GUIDE = 10
G_OPTIMAL_TYPE = 1               # VBA information 기준: GOT1
VIRTUAL_WAFER_RADIUS = 144000.0  # VBA virtual point keep 조건: R <= 144000               # VBA option D59: 1,2,3,4. 4 = inner OT2 / partial OT1
USE_SPAN_BOX_CONSTRAINT = True   # VBA Balloon projected CD/PGD stage

def cpe_38para(rx: Sequence[float], ry: Sequence[float]) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    X_dx = np.vstack([
        np.ones(len(rx)),
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
        (rx ** 3 * ry) / 1e19, (rx ** 2 * ry ** 2) / 1e19, (rx * ry ** 3) / 1e19, (ry ** 4) / 1e19,
        (rx ** 3 * ry ** 2) / 1e23, (rx ** 2 * ry ** 3) / 1e23, (rx * ry ** 4) / 1e23, (ry ** 5) / 1e23,
        (rx ** 3 * ry ** 3) / 1e27, (rx ** 2 * ry ** 4) / 1e27,
        (rx ** 3 * ry ** 4) / 1e31,
    ]).T
    X_dy = np.vstack([
        np.ones(len(ry)),
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12,
        (ry ** 4) / 1e19, (ry ** 3 * rx) / 1e19, (ry ** 2 * rx ** 2) / 1e19,
        (ry ** 5) / 1e23, (ry ** 4 * rx) / 1e23, (ry ** 3 * rx ** 2) / 1e23,
        (ry ** 5 * rx) / 1e27, (ry ** 4 * rx ** 2) / 1e27,
    ]).T
    return X_dx, X_dy
CPE38_KEYS_DX = [
    "RK1", "RK3", "RK5",
    "RK7", "RK9", "RK11",
    "RK13", "RK15", "RK17", "RK19",
    "RK23", "RK25", "RK27", "RK29",
    "RK35", "RK37", "RK39", "RK41",
    "RK49", "RK51",
    "RK65",
]
CPE38_KEYS_DY = [
    "RK2", "RK4", "RK6",
    "RK8", "RK10", "RK12",
    "RK14", "RK16", "RK18",
    "RK22", "RK24", "RK26",
    "RK32", "RK34", "RK36",
    "RK46", "RK48",
]
RK_SPAN_LIMITS_OVO3_38P_F2F = {
    'RK1': {'min': -0.1, 'max': 0.1},
    'RK2': {'min': -0.1, 'max': 0.1},
    'RK3': {'min': -2.0, 'max': 2.0},
    'RK4': {'min': -10.0, 'max': 10.0},
    'RK5': {'min': -5.0, 'max': 5.0},
    'RK6': {'min': -5.0, 'max': 5.0},
    'RK7': {'min': -0.1, 'max': 0.1},
    'RK8': {'min': -0.15, 'max': 0.15},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -0.5, 'max': 0.5},
    'RK11': {'min': -0.5, 'max': 0.5},
    'RK12': {'min': -0.03, 'max': 0.03},
    'RK13': {'min': -0.003, 'max': 0.003},
    'RK14': {'min': -0.015, 'max': 0.015},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -0.05, 'max': 0.05},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -0.023, 'max': 0.023},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}
RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT = {
    'RK1': {'min': -100.0, 'max': 100.0},
    'RK2': {'min': -100.0, 'max': 100.0},
    'RK3': {'min': -1000.0, 'max': 1000.0},
    'RK4': {'min': -1000.0, 'max': 1000.0},
    'RK5': {'min': -1000.0, 'max': 1000.0},
    'RK6': {'min': -1000.0, 'max': 1000.0},
    'RK7': {'min': -0.25, 'max': 0.25},
    'RK8': {'min': -10.0, 'max': 10.0},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -10.0, 'max': 10.0},
    'RK11': {'min': -10.0, 'max': 10.0},
    'RK12': {'min': -0.25, 'max': 0.25},
    'RK13': {'min': -0.025, 'max': 0.025},
    'RK14': {'min': -1.0, 'max': 1.0},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -1.0, 'max': 1.0},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -1.0, 'max': 1.0},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}

def get_active_span_limits(span_mode: str = "OVO3_38P_F2F") -> Dict[str, Dict[str, float]]:
    if span_mode == "OVO3_38P_HARDLIMIT":
        return RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT
    return RK_SPAN_LIMITS_OVO3_38P_F2F
RIDGE_VALVE_HYPER = {
    "RK1": 0.0, "RK2": 0.0,
    "RK3": 0.01, "RK4": 0.01, "RK5": 0.01, "RK6": 0.01,
    "RK7": 1.0, "RK8": 0.1, "RK9": 1.0, "RK10": 0.1,
    "RK11": 0.1, "RK12": 1.0,
    "RK13": 1.0, "RK14": 0.1, "RK15": 1.0, "RK16": 0.1,
    "RK17": 1.0, "RK18": 1.0, "RK19": 0.1, "RK20": 1.0,
}


## 23. CPE virtual grid 및 소표본 ridge 배율
chip별 네 모서리+중심점을 생성하고 중복을 제거한 뒤 wafer 반경 **144000 µm 이하**를 유지합니다. chip 정보가 없으면 측정 범위의 10×10 grid로 대체합니다. virtual point가 모두 반경 밖이면 원본처럼 full grid로 대체하며 `virtual_source`에 기록합니다. 이 반경은 TROCS 전체 mark 생성 반경 147000 µm와 다릅니다.


In [ ]:
def _to_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def choose_base_lambda_scale(n_points: int) -> float:
    """VBA의 point가 적어도 skip하지 않는 방식을 Python식으로 반영."""
    if n_points >= 10:
        return 1.0
    if n_points >= 7:
        return 1e2
    if n_points >= 5:
        return 1e4
    if n_points >= 3:
        return 1e6
    return 1e8

def get_ridge_valves(coeff_names: Sequence[str]) -> np.ndarray:
    return np.array([RIDGE_VALVE_HYPER.get(name, 1.0) for name in coeff_names], dtype=float)

def make_lambda_vector(base_lambda: float, coeff_names: Sequence[str]) -> np.ndarray:
    valves = get_ridge_valves(coeff_names)
    return float(base_lambda) * valves

def build_virtual_chip_points(
    step_pitch_x: float,
    step_pitch_y: float,
    chip_x_num: int,
    chip_y_num: int,
    round_decimals: int = 6,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA virtual grid 재현:
    chip별 4 corner + center point를 shot-local coordinate로 생성하고,
    chip 경계에서 겹치는 corner는 dedup한다.
    예: chip_x_num=2, chip_y_num=1
      - 각 chip 5 point => 10 raw points
      - 가운데 공유 corner 2개 중복 제거
      - 최종 8 virtual points
    coordinate_X/Y가 shot center 기준이라고 보고,
    전체 shot 영역은 [-STEP_PITCH_X/2, +STEP_PITCH_X/2],
                  [-STEP_PITCH_Y/2, +STEP_PITCH_Y/2] 로 둔다.
    """
    chip_x_num = int(chip_x_num) if pd.notna(chip_x_num) and int(chip_x_num) > 0 else 1
    chip_y_num = int(chip_y_num) if pd.notna(chip_y_num) and int(chip_y_num) > 0 else 1
    sx = float(step_pitch_x)
    sy = float(step_pitch_y)
    x_edges = np.linspace(-sx / 2.0, sx / 2.0, chip_x_num + 1)
    y_edges = np.linspace(-sy / 2.0, sy / 2.0, chip_y_num + 1)
    pts = []
    for ix in range(chip_x_num):
        for iy in range(chip_y_num):
            x0, x1 = x_edges[ix], x_edges[ix + 1]
            y0, y1 = y_edges[iy], y_edges[iy + 1]
            xc = (x0 + x1) / 2.0
            yc = (y0 + y1) / 2.0
            pts.extend([
                (x0, y0), (x0, y1), (x1, y0), (x1, y1),
                (xc, yc),
            ])
    pts_unique = sorted({(round(x, round_decimals), round(y, round_decimals)) for x, y in pts})
    rx = np.array([p[0] for p in pts_unique], dtype=float)
    ry = np.array([p[1] for p in pts_unique], dtype=float)
    return rx, ry

def filter_virtual_points_by_wafer_radius(
    rx_v: Sequence[float],
    ry_v: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    VBA edge-shot virtual point filtering.
    virtual point의 wafer 좌표 R이 wafer_radius 이하인 point만 사용한다.
    R = sqrt((fcp_x + rx_v)^2 + (fcp_y + ry_v)^2)
    keep if R <= 144000
    """
    rx_v = np.asarray(rx_v, dtype=float)
    ry_v = np.asarray(ry_v, dtype=float)
    wx = float(fcp_x) + rx_v
    wy = float(fcp_y) + ry_v
    r = np.sqrt(wx * wx + wy * wy)
    keep = r <= float(wafer_radius)
    return rx_v[keep], ry_v[keep], r[keep]

def get_virtual_points_for_group(
    grp: pd.DataFrame,
    fallback_rx: Sequence[float],
    fallback_ry: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, str, int, int]:
    """
    1) full virtual point 생성: chip별 4 corner + center
    2) edge shot 처리: wafer 좌표 R <= wafer_radius point만 keep
    반환:
      rx_v, ry_v, virtual_source, n_virtual_full, n_virtual_kept
    """
    required = ["STEP_PITCH_X", "STEP_PITCH_Y", "CHIP_X_NUM", "CHIP_Y_NUM"]
    if all(c in grp.columns for c in required):
        vals = grp[required].dropna()
        if len(vals) > 0:
            row = vals.iloc[0]
            rx_full, ry_full = build_virtual_chip_points(
                step_pitch_x=row["STEP_PITCH_X"],
                step_pitch_y=row["STEP_PITCH_Y"],
                chip_x_num=row["CHIP_X_NUM"],
                chip_y_num=row["CHIP_Y_NUM"],
            )
            n_full = len(rx_full)
            rx_keep, ry_keep, _ = filter_virtual_points_by_wafer_radius(
                rx_full,
                ry_full,
                fcp_x=fcp_x,
                fcp_y=fcp_y,
                wafer_radius=wafer_radius,
            )
            if len(rx_keep) > 0:
                src = "chip_corner_center_wafer_clipped"
                return rx_keep, ry_keep, src, n_full, len(rx_keep)
            return rx_full, ry_full, "chip_corner_center_no_clip_points_fallback", n_full, n_full
    rx_v, ry_v = build_eval_grid_from_data(fallback_rx, fallback_ry, grid_n=CPE_EVAL_GRID_N)
    return rx_v, ry_v, "measured_range_grid_fallback", len(rx_v), len(rx_v)


## 24. Ridge 예측 및 G-opt 평가 함수
측정 matrix와 평가 matrix를 분리해 `XᵀX + diag(lambda)`를 계산합니다. 기본 G-opt Type1은 hat matrix 행 제곱합의 최대값에 제곱근을 취합니다. Type2는 **현재 코드의 signed max에 제곱근**을 취하는 동작을 유지합니다.


In [ ]:
def stable_inverse_or_pinv(A: np.ndarray) -> np.ndarray:
    try:
        return np.linalg.inv(A)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A)

def ridge_predict(X: np.ndarray, y: np.ndarray, lambdas: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X.T @ X + np.diag(lambdas)
    b = X.T @ y
    try:
        beta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(A) @ b
    return beta, X @ beta

def build_eval_grid_from_data(rx: Sequence[float], ry: Sequence[float], grid_n: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    rx_min, rx_max = np.nanmin(rx), np.nanmax(rx)
    ry_min, ry_max = np.nanmin(ry), np.nanmax(ry)
    if not np.isfinite(rx_min) or not np.isfinite(rx_max):
        rx_min, rx_max = -1.0, 1.0
    if not np.isfinite(ry_min) or not np.isfinite(ry_max):
        ry_min, ry_max = -1.0, 1.0
    if rx_min == rx_max:
        rx_min -= 1.0
        rx_max += 1.0
    if ry_min == ry_max:
        ry_min -= 1.0
        ry_max += 1.0
    gx = np.linspace(rx_min, rx_max, grid_n)
    gy = np.linspace(ry_min, ry_max, grid_n)
    RX, RY = np.meshgrid(gx, gy)
    return RX.ravel(), RY.ravel()

def gopt_value(
    X_meas: np.ndarray,
    X_eval: np.ndarray,
    lambdas: np.ndarray,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
) -> float:
    """
    VBA G_optimal_type 반영.
    - Type1: H H.T diag의 sqrt(max)
    - Type2: H matrix element의 signed max의 sqrt. 음수이면 999.0 반환.
    - Type3: X_eval inv(...) X_eval.T diag의 sqrt(max)
    - Type4: inner=Type2, partial=Type1
    """
    X_meas = np.asarray(X_meas, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X_meas.T @ X_meas + np.diag(lambdas)
    invA = stable_inverse_or_pinv(A)
    effective_type = gopt_type
    if gopt_type == 4:
        effective_type = 2 if inner_or_partial == "inner" else 1
    if effective_type == 1:
        H = X_eval @ invA @ X_meas.T
        diag_val = np.sum(H * H, axis=1)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    if effective_type == 2:
        H = X_eval @ invA @ X_meas.T
        max_val = float(np.nanmax(H))
        if max_val < 0:
            return 999.0
        return float(np.sqrt(max_val))
    if effective_type == 3:
        M = X_eval @ invA @ X_eval.T
        diag_val = np.diag(M)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    raise ValueError(f"Unsupported gopt_type: {gopt_type}")


## 25. Span/box 제약 및 한 방향 회귀 solver
현재 로직처럼 lambda=0으로 시작해 G-opt가 spec 이상/비유한/0이면 ridge를 확대하고 이분 탐색합니다. span 초과가 남으면 현재 lambda를 유지한 box coordinate descent를 실행합니다. 계수·제거값 계산을 원본과 비교할 수 있도록 solver의 수식과 반복 조건을 유지합니다.


In [ ]:
def calc_contribution_spans(X_eval: np.ndarray, beta: np.ndarray, coeff_names: Sequence[str]) -> Dict[str, float]:
    spans = {}
    for i, name in enumerate(coeff_names):
        contrib = X_eval[:, i] * beta[i]
        spans[name] = float(np.nanmax(contrib) - np.nanmin(contrib))
    return spans

def find_span_violations(
    spans: Dict[str, float],
    span_limits: Dict[str, Dict[str, float]],
    tolerance: float = 0.01,
) -> List[Tuple[str, float, float, float]]:
    violations = []
    for name, span in spans.items():
        if name not in span_limits:
            continue
        lo = span_limits[name]["min"] - tolerance
        hi = span_limits[name]["max"] + tolerance
        if span < lo or span > hi:
            violations.append((name, span, span_limits[name]["min"], span_limits[name]["max"]))
    return violations

def build_beta_bounds_from_span(
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA의 min_Span_X/Y, max_Span_X/Y에 대응하는 box constraint 근사.
    contribution span = range(X_j * beta_j)가 span_limit 안에 들어오도록
    beta_j bound를 계산한다.
    """
    n = len(coeff_names)
    lo = np.full(n, -np.inf, dtype=float)
    hi = np.full(n, np.inf, dtype=float)
    for j, name in enumerate(coeff_names):
        if name not in span_limits:
            continue
        lim = span_limits[name]
        span_abs = max(abs(float(lim["min"])), abs(float(lim["max"])))
        xj_range = float(np.nanmax(X_eval[:, j]) - np.nanmin(X_eval[:, j]))
        if xj_range <= 0 or not np.isfinite(xj_range):
            continue
        b = span_abs / xj_range
        lo[j] = -b
        hi[j] = b
    return lo, hi

def projected_coordinate_descent_ridge(
    X: np.ndarray,
    y: np.ndarray,
    lambdas: np.ndarray,
    beta_init: Optional[np.ndarray] = None,
    lower: Optional[np.ndarray] = None,
    upper: Optional[np.ndarray] = None,
    tol: float = 1e-6,
    max_iter: int = 5000,
) -> Tuple[np.ndarray, np.ndarray, int, float]:
    """
    VBA ProjectedCoordinateDescentElasticNet(l1=0) 근사.
    solve min ||y-Xb||^2 + sum(lambda_j*b_j^2), s.t. lower <= b <= upper
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    n_param = X.shape[1]
    if beta_init is None:
        beta, _ = ridge_predict(X, y, lambdas)
    else:
        beta = np.asarray(beta_init, dtype=float).copy()
    if lower is None:
        lower = np.full(n_param, -np.inf)
    if upper is None:
        upper = np.full(n_param, np.inf)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    beta = np.clip(beta, lower, upper)
    residual = y - X @ beta
    col_norm2 = np.sum(X * X, axis=0)
    last_delta = np.inf
    for it in range(1, max_iter + 1):
        max_delta = 0.0
        for j in range(n_param):
            denom = col_norm2[j] + lambdas[j]
            if denom <= 0 or not np.isfinite(denom):
                continue
            residual += X[:, j] * beta[j]
            rho = float(X[:, j] @ residual)
            new_beta = rho / denom
            new_beta = min(max(new_beta, lower[j]), upper[j])
            delta = abs(new_beta - beta[j])
            beta[j] = new_beta
            residual -= X[:, j] * beta[j]
            if delta > max_delta:
                max_delta = delta
        last_delta = max_delta
        if max_delta < tol:
            break
    return beta, X @ beta, it, float(last_delta)

@dataclass
class RidgeFitResult:
    beta: np.ndarray
    pred: np.ndarray
    lambdas: np.ndarray
    gopt: float
    spans: Dict[str, float]
    violations: List[Tuple[str, float, float, float]]
    gopt_iter: int
    span_iter: int
    method: str

def solve_vba_style_ridge_one_axis(
    X_fit: np.ndarray,
    y: np.ndarray,
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
    ridge_regression: bool = True,
    gopt_spec: float = 1.0,
    gopt_tolerance: float = 0.001,
    minimum_ridge: float = 0.001,
    add_ridge: float = 10.0,
    max_gopt_return: int = 50,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
    use_span_box_constraint: bool = True,
    span_tolerance: float = 0.01,
    cd_tol: float = 1e-6,
    max_span_iter: int = 5000,
    base_lambda_scale: float = 1.0,
    debug: bool = False,
) -> RidgeFitResult:
    """
    VBA flow에 맞춘 한 방향(X or Y) fitting.
    - 초기 ridge는 0에 가까운 값이 아니라 VBA minimum_Ridge 기준 사용
    - Gopt가 spec 이상이면 ridge를 빠르게 키운 뒤 이분탐색으로 spec 근처를 찾음
    - span over가 남으면 projected coordinate descent로 coefficient를 box constraint 안에 넣음
    """
    X_fit = np.asarray(X_fit, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    y = np.asarray(y, dtype=float)
    n_param = X_fit.shape[1]
    if n_param != len(coeff_names):
        raise ValueError(f"n_param({n_param}) != len(coeff_names)({len(coeff_names)})")
    initial_lambda = 0.0 if not ridge_regression else 0.0
    lambdas = make_lambda_vector(initial_lambda, coeff_names)
    beta, pred = ridge_predict(X_fit, y, lambdas)
    g = gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
    gopt_iter = 0
    method = "ols"
    if ridge_regression and (not np.isfinite(g) or g >= gopt_spec or g == 0):
        low_lambda = 0.0
        high_lambda = max(minimum_ridge * base_lambda_scale, minimum_ridge)
        high_lambda = max(high_lambda, minimum_ridge)
        high_lambdas = make_lambda_vector(high_lambda, coeff_names)
        high_g = gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        while (not np.isfinite(high_g) or high_g >= gopt_spec) and gopt_iter < max_gopt_return:
            low_lambda = high_lambda
            high_lambda = high_lambda * add_ridge
            high_lambdas = make_lambda_vector(high_lambda, coeff_names)
            high_g = gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
            gopt_iter += 1
            if debug:
                print(f"fast ridge: lambda={high_lambda:.6g}, gopt={high_g:.6g}")
        target_low = max(0.0, gopt_spec - gopt_tolerance)
        target_high = gopt_spec
        best_lambda = high_lambda
        best_g = high_g
        if np.isfinite(high_g) and high_g < gopt_spec:
            lo = low_lambda
            hi = high_lambda
            for _ in range(max_gopt_return):
                mid = (lo + hi) / 2.0
                mid_lambdas = make_lambda_vector(mid, coeff_names)
                mid_g = gopt_value(X_fit, X_eval, mid_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
                gopt_iter += 1
                if debug:
                    print(f"slow ridge: lambda={mid:.6g}, gopt={mid_g:.6g}")
                if target_low <= mid_g < target_high:
                    best_lambda = mid
                    best_g = mid_g
                    break
                if mid_g >= gopt_spec or not np.isfinite(mid_g):
                    lo = mid
                else:
                    hi = mid
                    best_lambda = mid
                    best_g = mid_g
                if abs(hi - lo) < max(minimum_ridge * 1e-3, 1e-12):
                    break
        lambdas = make_lambda_vector(best_lambda, coeff_names)
        beta, pred = ridge_predict(X_fit, y, lambdas)
        g = gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt"
    spans = calc_contribution_spans(X_eval, beta, coeff_names)
    violations = find_span_violations(spans, span_limits, tolerance=span_tolerance)
    span_iter = 0
    if ridge_regression and use_span_box_constraint and len(violations) > 0:
        lower, upper = build_beta_bounds_from_span(X_eval, coeff_names, span_limits)
        beta_cd, pred_cd, span_iter, _ = projected_coordinate_descent_ridge(
            X_fit,
            y,
            lambdas,
            beta_init=beta,
            lower=lower,
            upper=upper,
            tol=cd_tol,
            max_iter=max_span_iter,
        )
        beta = beta_cd
        pred = pred_cd
        spans = calc_contribution_spans(X_eval, beta, coeff_names)
        violations = find_span_violations(spans, span_limits, tolerance=span_tolerance)
        g = gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt_box_cd"
    return RidgeFitResult(
        beta=beta,
        pred=pred,
        lambdas=lambdas,
        gopt=float(g),
        spans=spans,
        violations=violations,
        gopt_iter=int(gopt_iter),
        span_iter=int(span_iter),
        method=method,
    )

def determine_inner_or_partial(n_valid: int, n_virtual: int) -> str:
    """VBA: virtual_points = Vgrid_XY면 inner, 아니면 partial."""
    return "inner" if n_valid >= n_virtual else "partial"


## 26. Shot별 CPE38 회귀 함수
`(LOT_ID, Wafer, fcp_x, fcp_y)`별로 WK/RK 제거 잔차를 회귀합니다. 유효 point가 1개 이상이면 원본처럼 소표본 ridge 배율로 계산하고, 0개이면 `no_valid_points`로 기록합니다.

원본 결과에 **각 RK의 실제 lambda 값**만 추가로 저장합니다. shot 계수에는 RK*_X/Y, span_RK*, lambda_RK*, G-opt, 반복 수, virtual point 정보가 포함됩니다. 원본의 `status="ok"`는 계산 완료 의미이므로 27에서 spec 충족 여부를 별도 표시합니다.


In [ ]:
def run_cpe38_vba_style_shotwise(
    df: pd.DataFrame,
    label: str = "ADI",
    wafer_cols: Tuple[str, str] = ("LOT_ID", "Wafer"),
    shot_center_cols: Tuple[str, str] = ("fcp_x", "fcp_y"),
    residual_cols: Tuple[str, str] = ("raw_minus_total_fit_x", "raw_minus_total_fit_y"),
    shot_cols: Tuple[str, str] = ("coordinate_X", "coordinate_Y"),
    span_mode: str = SPAN_MODE,
    min_points_guide: int = MIN_POINTS_GUIDE,
    eval_grid_n: int = CPE_EVAL_GRID_N,
    ridge_regression: bool = RIDGE_REGRESSION,
    gopt_spec: float = G_OPT_SPEC,
    gopt_tolerance: float = G_OPT_TOLERANCE,
    gopt_type: int = G_OPTIMAL_TYPE,
    use_span_box_constraint: bool = USE_SPAN_BOX_CONSTRAINT,
    debug: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    기존 018.py의 run_cpe38_ridge_shotwise_after_wkrk 대체 함수.
    Required input columns:
      LOT_ID, Wafer, fcp_x, fcp_y,
      raw_minus_total_fit_x, raw_minus_total_fit_y,
      coordinate_X, coordinate_Y
    """
    df = df.copy()
    required = list(wafer_cols) + list(shot_center_cols) + list(residual_cols) + list(shot_cols)
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"[{label}] 필수 컬럼 누락: {missing}")
    _to_numeric(df, list(shot_center_cols) + list(residual_cols) + list(shot_cols))
    out_cols = [
        "CPE_fit_x", "CPE_fit_y",
        "real_residual_x", "real_residual_y",
        "CPE_gopt_x", "CPE_gopt_y",
        "CPE_lambda_max_x", "CPE_lambda_max_y",
        "CPE_method_x", "CPE_method_y",
        "CPE_inner_or_partial",
    ]
    for c in out_cols:
        df[c] = np.nan if not c.startswith("CPE_method") and c != "CPE_inner_or_partial" else None
    span_limits = get_active_span_limits(span_mode)
    coef_rows = []
    summary_rows = []
    group_keys = list(wafer_cols) + list(shot_center_cols)
    for gkey, grp in df.groupby(group_keys, dropna=False):
        valid_mask = (
            grp[residual_cols[0]].notna() &
            grp[residual_cols[1]].notna() &
            grp[shot_cols[0]].notna() &
            grp[shot_cols[1]].notna()
        )
        work = grp.loc[valid_mask].copy()
        n_valid = len(work)
        lot_id = gkey[0]
        wafer = gkey[1]
        fcp_x = gkey[2]
        fcp_y = gkey[3]
        if n_valid == 0:
            summary_rows.append({
                "LOT_ID": lot_id,
                "Wafer": wafer,
                "fcp_x": fcp_x,
                "fcp_y": fcp_y,
                "n_total": len(grp),
                "n_valid": 0,
                "few_points_flag": True,
                "status": "no_valid_points",
            })
            continue
        rx = work[shot_cols[0]].to_numpy(dtype=float)
        ry = work[shot_cols[1]].to_numpy(dtype=float)
        residual_x = work[residual_cols[0]].to_numpy(dtype=float)
        residual_y = work[residual_cols[1]].to_numpy(dtype=float)
        X_dx, X_dy = cpe_38para(rx, ry)
        if USE_VIRTUAL_CHIP_POINTS:
            grid_rx, grid_ry, virtual_source, n_virtual_full, n_virtual = get_virtual_points_for_group(
                work,
                rx,
                ry,
                fcp_x=fcp_x,
                fcp_y=fcp_y,
                wafer_radius=VIRTUAL_WAFER_RADIUS,
            )
        else:
            grid_rx, grid_ry = build_eval_grid_from_data(rx, ry, grid_n=eval_grid_n)
            virtual_source = "measured_range_grid"
            n_virtual_full = len(grid_rx)
            n_virtual = len(grid_rx)
        X_eval_dx, X_eval_dy = cpe_38para(grid_rx, grid_ry)
        inner_or_partial = determine_inner_or_partial(n_valid=n_valid, n_virtual=n_virtual)
        few_points_flag = n_valid < min_points_guide
        base_lambda_scale = choose_base_lambda_scale(n_valid)
        fit_x = solve_vba_style_ridge_one_axis(
            X_fit=X_dx,
            y=residual_x,
            X_eval=X_eval_dx,
            coeff_names=CPE38_KEYS_DX,
            span_limits=span_limits,
            ridge_regression=ridge_regression,
            gopt_spec=gopt_spec,
            gopt_tolerance=gopt_tolerance,
            minimum_ridge=MINIMUM_RIDGE,
            add_ridge=ADD_RIDGE,
            max_gopt_return=MAX_GOPT_RETURN,
            gopt_type=gopt_type,
            inner_or_partial=inner_or_partial,
            use_span_box_constraint=use_span_box_constraint,
            span_tolerance=SPAN_TOLERANCE,
            cd_tol=CD_TOL,
            max_span_iter=MAX_SPAN_ITER,
            base_lambda_scale=base_lambda_scale,
            debug=debug,
        )
        fit_y = solve_vba_style_ridge_one_axis(
            X_fit=X_dy,
            y=residual_y,
            X_eval=X_eval_dy,
            coeff_names=CPE38_KEYS_DY,
            span_limits=span_limits,
            ridge_regression=ridge_regression,
            gopt_spec=gopt_spec,
            gopt_tolerance=gopt_tolerance,
            minimum_ridge=MINIMUM_RIDGE,
            add_ridge=ADD_RIDGE,
            max_gopt_return=MAX_GOPT_RETURN,
            gopt_type=gopt_type,
            inner_or_partial=inner_or_partial,
            use_span_box_constraint=use_span_box_constraint,
            span_tolerance=SPAN_TOLERANCE,
            cd_tol=CD_TOL,
            max_span_iter=MAX_SPAN_ITER,
            base_lambda_scale=base_lambda_scale,
            debug=debug,
        )
        cpe_fit_x = fit_x.pred
        cpe_fit_y = fit_y.pred
        real_residual_x = residual_x - cpe_fit_x
        real_residual_y = residual_y - cpe_fit_y
        valid_idx = work.index
        df.loc[valid_idx, "CPE_fit_x"] = cpe_fit_x
        df.loc[valid_idx, "CPE_fit_y"] = cpe_fit_y
        df.loc[valid_idx, "real_residual_x"] = real_residual_x
        df.loc[valid_idx, "real_residual_y"] = real_residual_y
        df.loc[valid_idx, "CPE_gopt_x"] = fit_x.gopt
        df.loc[valid_idx, "CPE_gopt_y"] = fit_y.gopt
        df.loc[valid_idx, "CPE_lambda_max_x"] = float(np.nanmax(fit_x.lambdas))
        df.loc[valid_idx, "CPE_lambda_max_y"] = float(np.nanmax(fit_y.lambdas))
        df.loc[valid_idx, "CPE_method_x"] = fit_x.method
        df.loc[valid_idx, "CPE_method_y"] = fit_y.method
        df.loc[valid_idx, "CPE_inner_or_partial"] = inner_or_partial
        coef_row = {
            "LOT_ID": lot_id,
            "Wafer": wafer,
            "fcp_x": fcp_x,
            "fcp_y": fcp_y,
            "n_total": len(grp),
            "n_valid": n_valid,
            "n_virtual": n_virtual,
            "n_virtual_full": n_virtual_full,
            "virtual_source": virtual_source,
            "inner_or_partial": inner_or_partial,
            "few_points_flag": few_points_flag,
            "base_lambda_scale": base_lambda_scale,
            "gopt_x": fit_x.gopt,
            "gopt_y": fit_y.gopt,
            "lambda_max_x": float(np.nanmax(fit_x.lambdas)),
            "lambda_max_y": float(np.nanmax(fit_y.lambdas)),
            "gopt_iter_x": fit_x.gopt_iter,
            "gopt_iter_y": fit_y.gopt_iter,
            "span_iter_x": fit_x.span_iter,
            "span_iter_y": fit_y.span_iter,
            "span_violation_count_x": len(fit_x.violations),
            "span_violation_count_y": len(fit_y.violations),
            "method_x": fit_x.method,
            "method_y": fit_y.method,
        }
        for name, val in zip(CPE38_KEYS_DX, fit_x.beta):
            coef_row[f"{name}_X"] = val
            coef_row[f"span_{name}_X"] = fit_x.spans.get(name, np.nan)
        for name, val in zip(CPE38_KEYS_DY, fit_y.beta):
            coef_row[f"{name}_Y"] = val
            coef_row[f"span_{name}_Y"] = fit_y.spans.get(name, np.nan)
        for name, value in zip(CPE38_KEYS_DX, fit_x.lambdas):
            coef_row[f"lambda_{name}_X"] = float(value)
        for name, value in zip(CPE38_KEYS_DY, fit_y.lambdas):
            coef_row[f"lambda_{name}_Y"] = float(value)
        coef_rows.append(coef_row)
        summary_rows.append({
            "LOT_ID": lot_id,
            "Wafer": wafer,
            "fcp_x": fcp_x,
            "fcp_y": fcp_y,
            "n_total": len(grp),
            "n_valid": n_valid,
            "n_virtual": n_virtual,
            "n_virtual_full": n_virtual_full,
            "virtual_source": virtual_source,
            "inner_or_partial": inner_or_partial,
            "metro_ratio": n_valid / max(n_virtual, 1),
            "few_points_flag": few_points_flag,
            "base_lambda_scale": base_lambda_scale,
            "gopt_x": fit_x.gopt,
            "gopt_y": fit_y.gopt,
            "lambda_max_x": float(np.nanmax(fit_x.lambdas)),
            "lambda_max_y": float(np.nanmax(fit_y.lambdas)),
            "gopt_iter_x": fit_x.gopt_iter,
            "gopt_iter_y": fit_y.gopt_iter,
            "span_iter_x": fit_x.span_iter,
            "span_iter_y": fit_y.span_iter,
            "span_violation_count_x": len(fit_x.violations),
            "span_violation_count_y": len(fit_y.violations),
            "rms_before_x": float(np.sqrt(np.nanmean(residual_x ** 2))),
            "rms_before_y": float(np.sqrt(np.nanmean(residual_y ** 2))),
            "rms_after_x": float(np.sqrt(np.nanmean(real_residual_x ** 2))),
            "rms_after_y": float(np.sqrt(np.nanmean(real_residual_y ** 2))),
            "method_x": fit_x.method,
            "method_y": fit_y.method,
            "status": "ok",
        })
    coef_df = pd.DataFrame(coef_rows)
    summary_df = pd.DataFrame(summary_rows)
    print(f"[{label}] VBA-style CPE38 ridge 완료")
    print(f"  shot groups: {len(summary_df)}")
    print(f"  coef rows: {len(coef_df)}")
    if "few_points_flag" in summary_df.columns:
        print(f"  few points shots: {int(summary_df['few_points_flag'].sum())}")
    if "span_violation_count_x" in summary_df.columns:
        print(f"  remaining span violations X/Y: "
              f"{int(summary_df['span_violation_count_x'].sum())}/"
              f"{int(summary_df['span_violation_count_y'].sum())}")
    return df, coef_df, summary_df


## 27. CPE 제거 실행, shot RK 계수 및 wafer/shot 결과 저장
`real_residual_x/y = raw_minus_total_fit_x/y - CPE_fit_x/y`가 최종 residual입니다. `CPE_SHOT_COEFFICIENTS`에 **shot별 제거한 RK 계수**를, `CPE_REMOVED_POINTS`에 point별 CPE 제거값을 저장합니다. `FINAL_POINTS`는 기존 측정값부터 최종 residual까지 모두 포함합니다.

`CPE_SHOT_DIAGNOSTICS`는 회귀 상태/G-opt/span 진단, `CPE_WAFER_DATA`와 `CPE_SHOT_DATA`는 CPE 입력·제거값·최종 residual 통계입니다. 진단의 `cpe_quality_ok`는 이번 회귀의 수치 조건 충족 여부이며 wafer 양/불 또는 rework 판정이 아닙니다.


In [ ]:
def cpe_quality_diagnostics(summary):
    out = summary.reindex(columns=list(dict.fromkeys(SHOT_KEYS + list(summary.columns)))).copy()
    status = out["status"] if "status" in out else pd.Series(index=out.index, dtype="object")
    for axis in ["x", "y"]:
        gopt = pd.to_numeric(out.get(f"gopt_{axis}", pd.Series(index=out.index, dtype=float)), errors="coerce")
        count = pd.to_numeric(out.get(f"span_violation_count_{axis}", pd.Series(index=out.index, dtype=float)), errors="coerce")
        out[f"gopt_within_spec_{axis}"] = np.isfinite(gopt) & (gopt > 0) & (gopt < G_OPT_SPEC)
        out[f"span_within_limits_{axis}"] = count.eq(0)
    flags = [f"{rule}_{axis}" for rule in ["gopt_within_spec", "span_within_limits"] for axis in ["x", "y"]]
    out["cpe_quality_ok"] = status.eq("ok") & out[flags].all(axis=1)
    out["quality_status"] = np.where(status.ne("ok"), "no_valid_points", np.where(out["cpe_quality_ok"], "within_current_specs", "review_gopt_or_span"))
    return out

cpe_options = dict(span_mode=SPAN_MODE, min_points_guide=MIN_POINTS_GUIDE, eval_grid_n=CPE_EVAL_GRID_N, ridge_regression=RIDGE_REGRESSION, gopt_spec=G_OPT_SPEC, gopt_tolerance=G_OPT_TOLERANCE, gopt_type=G_OPTIMAL_TYPE, use_span_box_constraint=USE_SPAN_BOX_CONSTRAINT)
df_adi_cpe, coef_adi_cpe, summary_adi_cpe = run_cpe38_vba_style_shotwise(df_adi_reg, label="ADI", **cpe_options)
df_oco_cpe, coef_oco_cpe, summary_oco_cpe = run_cpe38_vba_style_shotwise(df_oco_reg, label="OCO", **cpe_options)
cpe_coef_columns = SHOT_KEYS + [f"{prefix}{key}_{axis}" for prefix in ["", "span_", "lambda_"] for axis, keys in [("X", CPE38_KEYS_DX), ("Y", CPE38_KEYS_DY)] for key in keys]
coef_adi_cpe = coef_adi_cpe.reindex(columns=list(dict.fromkeys(cpe_coef_columns + list(coef_adi_cpe.columns))))
coef_oco_cpe = coef_oco_cpe.reindex(columns=list(dict.fromkeys(cpe_coef_columns + list(coef_oco_cpe.columns))))
df_cpe_diag_adi, df_cpe_diag_oco = cpe_quality_diagnostics(summary_adi_cpe), cpe_quality_diagnostics(summary_oco_cpe)
cpe_values = ["raw_minus_total_fit_x", "raw_minus_total_fit_y", "CPE_fit_x", "CPE_fit_y", "real_residual_x", "real_residual_y"]
df_cpe_wafer_adi, df_cpe_wafer_oco = summarize_values(df_adi_cpe, WAFER_KEYS, cpe_values), summarize_values(df_oco_cpe, WAFER_KEYS, cpe_values)
df_cpe_shot_adi, df_cpe_shot_oco = summarize_values(df_adi_cpe, SHOT_KEYS, cpe_values), summarize_values(df_oco_cpe, SHOT_KEYS, cpe_values)
df_cpe_removed_adi = df_adi_cpe[[col for col in POINT_KEYS + ["CPE_fit_x", "CPE_fit_y"] if col in df_adi_cpe]].copy()
df_cpe_removed_oco = df_oco_cpe[[col for col in POINT_KEYS + ["CPE_fit_x", "CPE_fit_y"] if col in df_oco_cpe]].copy()
for mode, final, coeff, points, diag, wafer, shot in [("ADI", df_adi_cpe, coef_adi_cpe, df_cpe_removed_adi, df_cpe_diag_adi, df_cpe_wafer_adi, df_cpe_shot_adi), ("OCO", df_oco_cpe, coef_oco_cpe, df_cpe_removed_oco, df_cpe_diag_oco, df_cpe_wafer_oco, df_cpe_shot_oco)]:
    save_stage(f"27_{mode}_FINAL_POINTS", final, columns=POINT_KEYS + ["raw_x", "raw_y", "wk_fit_x", "wk_fit_y", "rk_fit_x", "rk_fit_y"] + cpe_values)
    save_stage(f"27_{mode}_CPE_SHOT_COEFFICIENTS", coeff)
    save_stage(f"27_{mode}_CPE_REMOVED_POINTS", points)
    save_stage(f"27_{mode}_CPE_SHOT_DIAGNOSTICS", diag)
    save_stage(f"27_{mode}_CPE_WAFER_DATA", wafer)
    save_stage(f"27_{mode}_CPE_SHOT_DATA", shot)


## 28. 최종 분해식 확인 및 전체 저장 목록
각 유효 point에 대해 `raw = WK_fit + RK_fit + CPE_fit + real_residual`을 확인합니다. 회귀가 생략된 점은 NaN을 유지하므로 통계의 `n_valid`와 진단 파일을 함께 확인하세요. 최종 wafer/shot 자료는 원래 raw, TROCS 연결값, 각 제거 성분, 최종 residual 통계를 모두 포함합니다.

`df_adi_cpe`/`df_oco_cpe`가 최종 point 결과이며, `15_*_FINAL`은 MRC/PSM decorrect 완료 시점의 기존 결과로 유지합니다. 보정값과 residual은 입력 `raw`와 같은 단위이고, 계수는 원본 행렬의 스케일을 유지합니다. 모든 좌표는 µm입니다. 측정값의 단위 변환을 추가하지 않습니다.


In [ ]:
def check_decomposition(frame, mode):
    checks = []
    for axis in ["x", "y"]:
        cols = [f"raw_{axis}", f"wk_fit_{axis}", f"rk_fit_{axis}", f"CPE_fit_{axis}", f"real_residual_{axis}"]
        valid = frame[cols].dropna()
        expected = valid[cols[1:]].sum(axis=1)
        error = (valid[cols[0]] - expected).abs()
        passed = np.isclose(valid[cols[0]], expected, rtol=1e-10, atol=1e-12)
        checks.append({"mode": mode, "axis": axis, "n_total": len(frame), "n_checked": len(valid), "n_unchecked": len(frame) - len(valid), "max_abs_error": float(error.max()) if len(error) else np.nan, "passed": bool(passed.all()) if len(valid) else False, "check_status": "checked" if len(valid) else "no_valid_points"})
        if len(valid) and not passed.all():
            raise ValueError(f"{mode}/{axis}: raw 분해식 검증 실패. 입력과 제거값을 확인하세요.")
    return pd.DataFrame(checks)

df_decomposition_check = pd.concat([check_decomposition(df_adi_cpe, "ADI"), check_decomposition(df_oco_cpe, "OCO")], ignore_index=True)
final_values = ["raw_x", "raw_y", "trocs_fit_x", "trocs_fit_y", "wk_fit_x", "wk_fit_y", "rk_fit_x", "rk_fit_y", "CPE_fit_x", "CPE_fit_y", "real_residual_x", "real_residual_y"]
df_final_wafer_adi, df_final_wafer_oco = summarize_values(df_adi_cpe, WAFER_KEYS, final_values), summarize_values(df_oco_cpe, WAFER_KEYS, final_values)
df_final_shot_adi, df_final_shot_oco = summarize_values(df_adi_cpe, SHOT_KEYS, final_values), summarize_values(df_oco_cpe, SHOT_KEYS, final_values)
save_stage("28_DECOMPOSITION_CHECK", df_decomposition_check)
save_stage("28_ADI_FINAL_WAFER_DATA", df_final_wafer_adi)
save_stage("28_OCO_FINAL_WAFER_DATA", df_final_wafer_oco)
save_stage("28_ADI_FINAL_SHOT_DATA", df_final_shot_adi)
save_stage("28_OCO_FINAL_SHOT_DATA", df_final_shot_oco)
extension_settings = dict(
    trocs_wafer_radius_um=TROCS_WAFER_RADIUS, trocs_subtracted_from_raw=False,
    trocs_query_file=str(TROCS_QUERY_FILE) if TROCS_QUERY_FILE is not None else None,
    wk_rk_min_points=WK_RK_MIN_POINTS, wk_rk_group_keys=WAFER_KEYS, cpe_group_keys=SHOT_KEYS, cpe_options=cpe_options,
    use_virtual_chip_points=USE_VIRTUAL_CHIP_POINTS, virtual_wafer_radius_um=VIRTUAL_WAFER_RADIUS,
    span_tolerance=SPAN_TOLERANCE, minimum_ridge=MINIMUM_RIDGE, add_ridge=ADD_RIDGE,
    max_gopt_return=MAX_GOPT_RETURN, max_span_iter=MAX_SPAN_ITER, cd_tol=CD_TOL,
    ridge_valves=RIDGE_VALVE_HYPER, active_span_limits=get_active_span_limits(SPAN_MODE),
)
config_path = OUTPUT_DIR / "extension_settings.json"
config_path.write_text(json.dumps(extension_settings, ensure_ascii=False, indent=2), encoding="utf-8")
display(FileLink(os.path.relpath(config_path, Path.cwd()), result_html_prefix="추가 단계 실행 설정: "))
df_saved_files = pd.DataFrame(stage_files.values())
display(df_saved_files)
display(FileLink(os.path.relpath(OUTPUT_DIR / "saved_files.csv", Path.cwd()), result_html_prefix="전체 저장 파일 목록: "))
source.save_manifest(cfg)
source.release_cache()
print("TROCS 연결 → WK/RK 제거 → shot CPE38 제거까지 완료했습니다.")


## 수정·재실행 메모 및 주요 결과 변수
- 대상 선정 변경: 00~02의 설정/조회부터 다시 실행합니다. replay는 저장 당시 SQL 조건과 일치해야 합니다.
- batch 선택 변경: 06부터, 좌표/metadata 변경: 08부터 다시 실행합니다.
- MRC 계산 변경: 10, 13~15 및 16 이후 의존 셀을 다시 실행합니다. PSM 또는 raw 변경: 13 또는 14부터 다시 실행합니다.
- TROCS 조회 변경: 16부터, wafer mark 배치/계산 변경: 17~19를 다시 실행합니다. 19의 실측 입력이 바뀌면 21, 27, 28도 다시 실행합니다.
- WK/RK 행렬/최소 point 수 변경: 20을 수정하고 21, 27, 28을 다시 실행합니다.
- CPE 설정/모델 변경: 22를 수정하고 필요하면 23~26 정의 셀 및 27, 28을 다시 실행합니다. 현재 옵션은 27 실행 시 명시적으로 전달합니다.
- 저장 파일 복원: `df_adi_reg = pd.read_parquet(STAGE_DIR / "21_ADI_WK_RK_REMOVED.parquet")`처럼 읽을 수 있습니다. 18의 요약/저장 helper와 22~26의 CPE 정의 셀을 준비하면 27부터 재계산할 수 있습니다.

| 구분 | ADI 주요 변수 | OCO 주요 변수 |
|---|---|---|
| MRC/PSM decorrect 결과 | `df_final_adi` | `df_final_oco` |
| TROCS 입력, 전체 mark | `df_trocs_input`, `df_trocs_allmarks_adi` | 같은 입력, `df_trocs_allmarks_oco` |
| TROCS 연결 실측 자료 | `df_adi_trocs` | `df_oco_trocs` |
| WK/RK 제거 결과 | `df_adi_reg` | `df_oco_reg` |
| wafer WK / RK 계수 | `df_wk_coeff_adi` / `df_rk_coeff_adi` | `df_wk_coeff_oco` / `df_rk_coeff_oco` |
| shot CPE38 RK 계수 | `coef_adi_cpe` | `coef_oco_cpe` |
| 최종 point 결과 | `df_adi_cpe` | `df_oco_cpe` |
| 최종 wafer / shot 통계 | `df_final_wafer_adi` / `df_final_shot_adi` | `df_final_wafer_oco` / `df_final_shot_oco` |

합성 입력에서 원본 `011~013.py`의 값과 비교하고 실제 Jupyter 커널에서 단계별 저장을 검증합니다. live DB와 생산 데이터 검증은 사내 환경에서 수행하세요. 이 노트북 자체가 실행 중 GitHub로 파일을 보내지는 않습니다.
